In [ ]:
# Code Carbon

from codecarbon import OfflineEmissionsTracker

# stop a tracker left running from a previous run of this cell
try:
    tracker.stop()
except NameError:
    pass          # no tracker defined yet (first run)
except Exception:
    pass          # partially-initialized/already-stopped tracker; ignore

tracker = OfflineEmissionsTracker(country_iso_code="USA", log_level="error", project_name="College Baseball XGBoost")
tracker.start()

In [ ]:
# Imports

import pandas as pd
import numpy as np
import re
import itertools
import xgboost as xgb
import json
import os
import pynvml
import shap
from difflib import get_close_matches, SequenceMatcher
from sklearn.model_selection import train_test_split
from scipy.special import expit
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
import plotly.express as px
from sklearn.inspection import PartialDependenceDisplay
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score, accuracy_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    precision_recall_curve, classification_report,
    confusion_matrix, mean_absolute_error, r2_score, brier_score_loss
)

# V10 - eligibility-gated, ratio-targeted, standardized figures

Three-stage XGBoost model for the NCAA-to-MLB draft. Stage 1 predicts whether a player is drafted,
Stage 2 where he goes among college players, Stage 3 what he signs for relative to his pick's slot
value.

Three things differ from earlier versions:

1. Draft eligibility is applied once in the preamble, so no stage trains on players who could not
   legally have been drafted.
2. Stage 3 predicts the bonus/slot ratio rather than dollars, since dollars are mostly determined
   by draft position.
3. Figures and metric tables come from shared helpers, so all three stages are reported the same
   way.

The held-out year is 2026 throughout, with leave-one-year-out results reported alongside it.

In [ ]:
# Settings

DATA_FILE = 'batting_pitching_combined_with_rpi_2026.csv'
JSON_DIR  = './MLBStatsAPIDraftDataAccess'

year_col       = 'year'
target_drafted = 'Drafted?'
target_pick    = 'Pick'

# Held-out year, pinned rather than taken from the data so it can't drift when a year is added
TEST_YEAR     = 2026
SIM_TEST_YEAR = 2025

# Draft eligibility: junior year completed, or 21 within ~45 days of the draft
AGE_ELIGIBLE           = 21
CLASS_ELIGIBLE_SEASONS = 3
APPLY_ELIGIBILITY      = True

# Keep players whose eligibility can't be determined (no age on record and under 3 seasons)
TREAT_UNKNOWN_AS_ELIGIBLE = False

SEASON_SOURCES = ['../ncaa_battingNoMinCSV/batting_combined_all.csv',
                  '../ncaa_pitchingNoMinCSV/pitching_combined_all.csv']

# Stage 1 train/test scheme and class handling
SPLIT_MODE   = 'temporal'   # 'temporal' | 'random'
BALANCE_MODE = 'none'       # 'none' | 'scale_pos_weight' | 'undersample'

# Stage 3 ratio target
RATIO_CLIP       = (0.01, 8.0)   # winsorize before logging, trimming only the pathological tail
AT_SLOT_BAND     = 0.02          # |log ratio| below this counts as signed at slot
POST10_ORDER_CUT = 200           # stand-in for round 11+ when the round isn't known yet

# Stage 3 two-part model: Part A predicts the spike at exactly slot, Part B regresses the rest.
# Part A needs a small feature set -- on all 134 features it overfits to chance.
PARTA_FEATURES = ['age', 'role', 'ip_pitch', 'so_pitch', 'era_pitch', 'fip_pitch', 'whip_pitch',
                  'k/9_pitch', 'pa_bat', 'wrc+_bat', 'woba_bat', 'ops_bat', 'rpi_team',
                  'api_weight', 'api_height', 'api_primaryPos']
PARTA_PARAMS = dict(n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8,
                    colsample_bytree=0.6, min_child_weight=10, reg_alpha=1.0, reg_lambda=5.0,
                    random_state=42)
PARTA_GATES = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]

# Shared model params. No early stopping anywhere -- there is no validation year to stop on.
RP_REG = dict(n_estimators=1000, learning_rate=0.03, max_depth=4, subsample=0.8,
              colsample_bytree=0.7, min_child_weight=5, reg_alpha=1.0, reg_lambda=5.0,
              random_state=42)
RP_CLF = dict(n_estimators=500, learning_rate=0.05, max_depth=6, subsample=0.8,
              colsample_bytree=0.8, random_state=42)

# Set True to re-run the Stage 1 grid search. Left off because the search found nothing better:
# its winner gained +0.0004 PR-AUC on the held-out year and lost 0.006 ROC-AUC.
RUN_HPO = False

SEEDS = [0, 1, 2, 3, 4]   # Stage 1 metrics are averaged over these

SAVE_FIGS = True
FIG_DIR   = 'figures'
FIG_DPI   = 200

# Kept on df for bookkeeping but never given to a model; asserted out of `features` below
NON_FEATURE_COLS = ['nameascii', 'playerid', year_col, target_drafted, target_pick, 'Round']

print(f"data={DATA_FILE}  TEST_YEAR={TEST_YEAR}  SIM_TEST_YEAR={SIM_TEST_YEAR}")
print(f"eligibility: age>={AGE_ELIGIBLE} or seasons>={CLASS_ELIGIBLE_SEASONS} | "
      f"apply={APPLY_ELIGIBILITY} | unknown_eligible={TREAT_UNKNOWN_AS_ELIGIBLE}")
print(f"stage 1: split={SPLIT_MODE} balance={BALANCE_MODE}")

In [ ]:
# Batting and pitching dataset with teams
df = pd.read_csv(DATA_FILE, delimiter=',')

In [ ]:
def calc_wp(record):
    if pd.isna(record):
        return np.nan
    
    # Extract all numbers from the string
    nums = re.findall(r'\d+', str(record))
    
    # Need at least wins and losses
    if len(nums) < 2:
        return np.nan
    
    wins = int(nums[0])
    losses = int(nums[1])
    
    total = wins + losses
    
    if total == 0:
        return np.nan
    
    return wins / total

columns = [
    'Conference_Record_team',
    'NC_Rec_team',
    'Home_team',
    'Road_team',
    'Neutral_team',
    'Q1_team',
    'Q2_team',
    'Q3_team',
    'Q4_team'
]

for col in columns:
    new_col = col.replace('_team', '') + '_WP_team'
    df[new_col] = df[col].apply(calc_wp)

In [ ]:
# list and drop columns that are less related to the target
# playerid is kept for the eligibility season counts, Round for the Stage 3 slot lookup
cols_to_drop = ['team_old', 'league_team', 'name', 'mlbamid', 'team', 'Acronym', 'Full Name_team', 'Full Team Name', 'Drafted By', 'Drafted From', 'team_new', 'division_team', 'id_team', 'team_teamstats', 'Difference_team', 'conf_national_rec_team', 'Conference_Record_team', 'Conference_Record_Wins_team', 'Conference_Record_Losses_team', 'NC_Rec_team', 'Home_team', 'Road_team', 'Neutral_team', 'Q1_team', 'Q2_team', 'Q3_team', 'Q4_team']
more_cols_to_drop = ['NC_Rec_Wins_team', 'NC_Rec_Losses_team', 'NC_RPI_team', 'NC_SOS_team', 'Home_Wins_team', 'Home_Losses_team', 'Road_Wins_team', 'Road_Losses_team', 'Neutral_Wins_team', 'Neutral_Losses_team', 'Q1_Wins_team', 'Q1_Losses_team', 'Q2_Wins_team', 'Q2_Losses_team', 'Q3_Wins_team', 'Q3_Losses_team', 'Q4_Wins_team', 'Q4_Losses_team']
cols_to_drop.extend(more_cols_to_drop)
df = df.drop(columns=cols_to_drop)
print(df.columns.tolist())

# convert the target to numerical values
df['Drafted?'] = df['Drafted?'].astype(int)


# Pythagorean Expectation Percent: ratio of expected win% (PE) to actual win% (WPCT)
# >1 = team underperformed (unlucky), <1 = team overperformed (lucky)
df['PE_pct_team'] = df['PE_team'] / df['WPCT_team'].replace(0, np.nan)

# Map role to numeric here rather than in the features cell, so df_all gets it too. Two-way
# players are labelled 'Two-Way' in this data, so both spellings are accepted.
ROLE_MAP = {'Pitcher': 0, 'Batter': 1, 'Two-Way': 2, 'Both': 2}
print("role values before mapping:", df['role'].value_counts(dropna=False).to_dict())
df['role'] = df['role'].map(ROLE_MAP)
assert df['role'].notna().all(), \
    f"unmapped role values: {df.loc[df['role'].isna(), 'role'].unique()}"
df['role'] = df['role'].astype(int)

---
## Draft eligibility

A four-year college player is draft-eligible once he has completed his junior year or turns 21
within about 45 days of the draft. Applying that after prediction, as earlier versions did, leaves
Stage 1 separating drafted players from a pool padded with freshmen and sophomores who could not
have been drafted at all.

The next four cells count each player's seasons and recover his age, label every player-season with
the reason it is (or isn't) eligible, print the resulting coverage, and then filter `df` once so all
three stages inherit the same population.

In [ ]:
# Count NCAA seasons per player and recover ages, using the no-minimum leaderboards as well as
# the modeling file -- the modeling file keeps only ~11 players per team, so early low-usage
# seasons are often missing from it

_pid = df['playerid'].astype(str).str.strip()

seasons_by_pid, births_by_pid = {}, {}

def _learn(pid, yr, age=None):
    if not pid or pid.lower() == 'nan' or pd.isna(yr):
        return
    seasons_by_pid.setdefault(pid, set()).add(int(yr))
    if age is not None and pd.notna(age):
        births_by_pid.setdefault(pid, []).append(int(yr) - int(age))

def _add_source(path):
    if not os.path.exists(path):
        print(f"  MISSING {path}")
        return
    d = pd.read_csv(path, encoding='utf-8-sig', low_memory=False)
    d.columns = [c.strip().lower() for c in d.columns]
    ycol = 'year' if 'year' in d.columns else ('season' if 'season' in d.columns else None)
    if 'playerid' not in d.columns or ycol is None:
        print(f"  SKIP {path} (no playerid/year)")
        return
    ages = d['age'] if 'age' in d.columns else pd.Series([None] * len(d))
    for pid, yr, age in zip(d['playerid'].astype(str).str.strip(), d[ycol], ages):
        _learn(pid, yr, age)
    print(f"  + {path}  ({len(d)} rows, age column: {'age' in d.columns})")

for _p in SEASON_SOURCES:
    _add_source(_p)
for _p, _y, _a in zip(_pid, df[year_col], df['age']):
    _learn(_p, _y, _a)

df['total_college_seasons'] = _pid.map(lambda p: len(seasons_by_pid.get(p, set())))

# Seasons completed as of this row's year. Using the career total instead would mark a freshman
# season eligible because the player later played three.
df['seasons_to_date'] = [sum(1 for y in seasons_by_pid.get(p, ()) if y <= yr)
                         for p, yr in zip(_pid, df[year_col])]
assert (df['seasons_to_date'] <= df['total_college_seasons']).all()
assert (df['seasons_to_date'] >= 1).all()

_spread = {p: max(v) - min(v) for p, v in births_by_pid.items()}
print(f"\nplayers with an implied birth year: {len(births_by_pid)}  |  "
      f"internally consistent: {sum(s == 0 for s in _spread.values())}  |  "
      f"off by >1 year: {sum(s > 1 for s in _spread.values())}")

# median, not min/max, so a future disagreeing source degrades gracefully
_byr = _pid.map(lambda p: np.median(births_by_pid[p]) if p in births_by_pid else np.nan)
df['age_filled'] = np.where(df['age'].notna(), df['age'], df[year_col] - _byr)
print(f"age nullness: raw {df['age'].isna().mean():.3f} -> filled {df['age_filled'].isna().mean():.3f}")
print(f"career seasons:  min {df['total_college_seasons'].min()} "
      f"median {df['total_college_seasons'].median():.0f} max {df['total_college_seasons'].max()}")
print(f"seasons_to_date: {df['seasons_to_date'].value_counts().sort_index().to_dict()}")

NON_FEATURE_COLS += ['total_college_seasons', 'seasons_to_date', 'age_filled']

The five values are evidence tiers, strongest first: `drafted` (MLB selected him, so
eligibility is a fact rather than an inference), `class` (3rd or later season as of that season),
`age` (21 or older), `ineligible` (age known, under 21, under 3 seasons) and `unknown` (no age on
record and under 3 seasons). `drafted` has to come first — it is what stops the filter deleting a
positive label. `class` and `age` are both sufficient on their own, so their order changes the
reason reported and never the verdict.

The simulation needs a second version without the `drafted` tier, since before a draft nobody knows
who was picked.

In [ ]:
# Label every player-season with the reason it is or isn't draft-eligible

def _basis(r):
    if r[target_drafted] == 1:
        return 'drafted'
    if r['seasons_to_date'] >= CLASS_ELIGIBLE_SEASONS:
        return 'class'
    if pd.notna(r['age_filled']) and r['age_filled'] >= AGE_ELIGIBLE:
        return 'age'
    if pd.isna(r['age_filled']):
        return 'unknown'
    return 'ineligible'

df['eligibility_basis'] = df.apply(_basis, axis=1)
_b = df['eligibility_basis']
df['eligible'] = _b.isin(['drafted', 'class', 'age']) | (TREAT_UNKNOWN_AS_ELIGIBLE & (_b == 'unknown'))

# Same rule without the drafted tier, for anything that has to stay pre-draft

def _basis_predraft(r):
    if r['seasons_to_date'] >= CLASS_ELIGIBLE_SEASONS:
        return 'class'
    if pd.notna(r['age_filled']) and r['age_filled'] >= AGE_ELIGIBLE:
        return 'age'
    if pd.isna(r['age_filled']):
        return 'unknown'
    return 'ineligible'

df['eligibility_basis_predraft'] = df.apply(_basis_predraft, axis=1)
_bp = df['eligibility_basis_predraft']
df['eligible_predraft'] = _bp.isin(['class', 'age']) | (TREAT_UNKNOWN_AS_ELIGIBLE & (_bp == 'unknown'))

NON_FEATURE_COLS += ['eligibility_basis', 'eligible',
                     'eligibility_basis_predraft', 'eligible_predraft']
print(df['eligibility_basis'].value_counts().to_string())
print(f"\neligible (training):  {int(df['eligible'].sum())}")
print(f"eligible_predraft:    {int(df['eligible_predraft'].sum())} "
      f"(drops the 'was drafted' evidence tier)")

In [ ]:
# Eligibility coverage by year, and the checks that the filter behaved

print("=" * 78)
print("ELIGIBILITY BY YEAR")
print("=" * 78)
display(pd.crosstab(df[year_col], df['eligibility_basis'], margins=True))

_k = df.groupby(year_col)['eligible'].agg(rows='size', kept='sum')
_k['dropped'] = _k['rows'] - _k['kept']
_k['dropped_%'] = (100 * _k['dropped'] / _k['rows']).round(1)
_k['pos_before'] = df.groupby(year_col)[target_drafted].sum()
_k['pos_after'] = df[df['eligible']].groupby(year_col)[target_drafted].sum()
_k['pos_lost'] = _k['pos_before'] - _k['pos_after']
_k['pos_rate_before_%'] = (100 * df.groupby(year_col)[target_drafted].mean()).round(2)
_k['pos_rate_after_%'] = (100 * df[df['eligible']].groupby(year_col)[target_drafted].mean()).round(2)
display(_k)

print(f"total: {int(_k['kept'].sum())} of {len(df)} rows kept "
      f"({_k['kept'].sum()/len(df):.1%}), {int(_k['dropped'].sum())} dropped, "
      f"{int(_k['pos_lost'].sum())} positives lost")

# Being drafted proves eligibility, so the filter must never remove a drafted player
assert (df.loc[~df['eligible'], target_drafted] == 0).all(), \
    "eligibility filter destroyed a positive label"

# What the other setting of the toggle would keep
_alt = _b.isin(['drafted', 'class', 'age']) | ((not TREAT_UNKNOWN_AS_ELIGIBLE) & (_b == 'unknown'))
print(f"\nwith TREAT_UNKNOWN_AS_ELIGIBLE={not TREAT_UNKNOWN_AS_ELIGIBLE}: "
      f"{int(_alt.sum())} rows would be kept ({_alt.mean():.1%}), "
      f"{int((df[target_drafted] & ~_alt).sum())} positives lost")
print(f"'unknown' rows by year: "
      f"{df[_b == 'unknown'].groupby(year_col).size().to_dict()}")

# What the career-total season count would have wrongly kept, for comparison
_career_class = df['total_college_seasons'] >= CLASS_ELIGIBLE_SEASONS
_would_keep = (df['eligibility_basis'] == 'ineligible') & _career_class
print(f"\nper-season vs career-total season count:")
print(f"  rows the career total would have wrongly kept: {int(_would_keep.sum())}")
print(f"    their age_filled:       {df.loc[_would_keep, 'age_filled'].value_counts().sort_index().to_dict()}")
print(f"    their seasons_to_date:  {df.loc[_would_keep, 'seasons_to_date'].value_counts().sort_index().to_dict()}")
print(f"    drafted among them:     {int(df.loc[_would_keep, target_drafted].sum())}  (expect 0)")

# The detector's errors are one-sided: a drafted underclassman is rescued by the drafted tier,
# an equally eligible undrafted one cannot be, so age is where spurious signal could hide
_rescued = df[target_drafted].astype(bool) & ~df['eligible_predraft']
_dropped_neg = (~df[target_drafted].astype(bool)) & ~df['eligible']
print("\n" + "-" * 78)
print("ASYMMETRY CAVEAT (one-sided error in the eligibility detector)")
print("-" * 78)
print(f"  positives retained only by the 'drafted' tier: {int(_rescued.sum())} "
      f"({_rescued.sum()/max(df[target_drafted].sum(),1):.1%} of positives)")
print(f"    their age_filled: {df.loc[_rescued, 'age_filled'].value_counts().sort_index().to_dict()}")
print(f"    their seasons-to-date: {df.loc[_rescued, 'seasons_to_date'].value_counts().sort_index().to_dict()}")
print(f"  negatives dropped as provably ineligible: {int(_dropped_neg.sum())}")
print(f"    their age_filled: {df.loc[_dropped_neg, 'age_filled'].value_counts().sort_index().to_dict()}")
print("  => within the age-20 / under-3-season group the filter is not label-symmetric.")

In [ ]:
# Filter df once here so every stage inherits it. df_all keeps the unfiltered population, which
# the simulation's eligibility audit needs to report what the gate removed

df_all = df.copy()
if APPLY_ELIGIBILITY:
    df = df[df['eligible']].copy()

print(f"modeling frame: {len(df)} of {len(df_all)} rows ({len(df)/len(df_all):.1%})")
print(f"  drafted: {int(df[target_drafted].sum())} of {int(df_all[target_drafted].sum())}")
print(f"  by year: {df.groupby(year_col).size().to_dict()}")

In [ ]:
pd.options.display.max_columns = None
df.head(10)

In [ ]:
df.info()

In [ ]:
player_features = list(df[['age', 'role', 'w_pitch', 'l_pitch', 'era_pitch', 'g_pitch', 'gs_pitch', 
                           'cg_pitch', 'sho_pitch', 'sv_pitch', 'ip_pitch', 'tbf_pitch', 'h_pitch', 
                           'r_pitch', 'er_pitch', 'hr_pitch', 'bb_pitch', 'hbp_pitch', 'wp_pitch', 
                           'bk_pitch', 'so_pitch', 'k/9_pitch', 'bb/9_pitch', 'k/bb_pitch', 
                           'hr/9_pitch', 'k%_pitch', 'bb%_pitch', 'k-bb%_pitch', 'avg_pitch', 
                           'whip_pitch', 'babip_pitch', 'lob%_pitch', 'fip_pitch', 'e-f_pitch', 
                           'g_bat', 'ab_bat', 'pa_bat', 'h_bat', '1b_bat', '2b_bat', '3b_bat', 
                           'hr_bat', 'r_bat', 'rbi_bat', 'bb_bat', 'so_bat', 'hbp_bat', 'sf_bat', 
                           'sh_bat', 'gdp_bat', 'sb_bat', 'cs_bat', 'avg_bat', 'bb%_bat', 'k%_bat', 
                           'bb/k_bat', 'obp_bat', 'slg_bat', 'ops_bat', 'iso_bat', 'spd_bat', 
                           'babip_bat', 'wsb_bat', 'wrc_bat', 'wraa_bat', 'woba_bat', 'wrc+_bat']].columns)

team_features = list(df[['conf_rpi_team', 'conf_rank_team', 'conf_national_wp_team',
                         'W_team', 'L_team', 'T_team', 'G_team', 'WPCT_team', 'PE_team', 'BB (Batting)_team', 
                         'AB_team', 'H_team', 'BA_team', 'DP_team', 'DPPG_team', '2B_team', '2BPG_team', 
                         'IP_team', 'R (Pitching)_team', 'ER_team', 'ERA_team', 'PO_team', 'A_team', 
                         'E_team', 'FPCT_team', 'HB_team', 'HBP_team', 'HA_team', 'HAPG_team', 'HR_team', 
                         'HRPG_team', 'SF_team', 'SH_team', 'OBP_team', 'SB_team', 'SBPG_team', 'CS_team', 
                         'R (Batting)_team', 'RPG_team', 'SHO_team', 'TB_team', 'SLG_team', 'SO_team', 
                         'BB (Pitching)_team', 'K/BB_team', 'K/9_team', 'TP_team', '3B_team', '3BPG_team', 
                         'WHIP_team', 'BBPG (Pitching)_team', 'PE_pct_team']].columns)

rpi_team_features = ['rpi_team', 'SOS_team', 'Conference_Record_WP_team', 'NC_Rec_WP_team', 'Home_WP_team', 'Road_WP_team', 'Neutral_WP_team', 'Q1_WP_team', 'Q2_WP_team', 'Q3_WP_team', 'Q4_WP_team']

features = player_features + team_features + rpi_team_features


# Playing-time ratios. The raw file keeps counting totals and team totals apart, so without
# these the model has to learn the division itself.
def add_usage_features(d):
    made = []

    def ratio(name, num, den):
        d[name] = d[num] / d[den].replace(0, np.nan)
        made.append(name)

    ratio('ip_per_g_pitch', 'ip_pitch', 'g_pitch')      # innings per appearance
    ratio('start_share_pitch', 'gs_pitch', 'g_pitch')   # share of appearances that were starts
    ratio('pa_per_g_bat', 'pa_bat', 'g_bat')            # plate appearances per game
    ratio('g_share_bat', 'g_bat', 'G_team')             # share of the team's games played
    ratio('ip_share_pitch', 'ip_pitch', 'IP_team')      # share of the team's innings thrown
    ratio('ab_share_bat', 'ab_bat', 'AB_team')          # share of the team's at-bats taken
    ratio('so_per_ip_pitch', 'so_pitch', 'ip_pitch')    # strikeouts on an innings basis
    return made


usage_features = add_usage_features(df)
add_usage_features(df_all)
features = features + usage_features
print(f"added {len(usage_features)} usage features: {usage_features}")

# The eligibility columns are bookkeeping, not predictors
_leak = sorted(set(features) & set(NON_FEATURE_COLS))
assert not _leak, f"non-feature columns leaked into features: {_leak}"
for _c in ('age_filled', 'eligible', 'eligibility_basis', 'total_college_seasons',
           'seasons_to_date'):
    assert _c not in features, f"{_c} must not be a feature"
print(f"{len(features)} features "
      f"({len(player_features)} player + {len(team_features)} team + "
      f"{len(rpi_team_features)} rpi + {len(usage_features)} usage)")
X = df[features]
y = df['Drafted?'].astype(int)

In [ ]:
X.head(10)

---
## Figure and metric helpers

One place for figure styling, filenames and the metric printers, so all three stages are plotted
and reported the same way. Figures are written to `figures/s{stage}_{name}.{png,pdf}`.

Not every plot suits every stage. Feature importance, partial dependence, SHAP and parallel
coordinates apply everywhere; the PR curve, calibration and score histogram need a probability so
they are Stage 1 only; predicted-vs-actual and residuals need a continuous target so they are
Stages 2 and 3, with Stage 2 ranked first because its raw output has no natural scale.

In [ ]:
# Had AI help build these helpers from the poster figures (same colors, font sizes, etc.)

NAVY, RED, GRAY = '#1a2a4a', '#c8102e', '#d7d7d7'
STAGE_TITLE = {1: 'Stage 1 - Draft Classifier',
               2: 'Stage 2 - College Draft Order',
               3: 'Stage 3 - Bonus / Slot Ratio'}

# raw column name -> display label
BASE_OVERRIDES = {
    'rpi': 'RPI', 'conf_rpi': 'Conf. RPI', 'sos': 'SOS',
    'wrc': 'wRC', 'wrc+': 'wRC+', 'woba': 'wOBA', 'wraa': 'wRAA', 'wsb': 'wSB',
    'ops': 'OPS', 'slg': 'SLG', 'obp': 'OBP', 'iso': 'ISO', 'avg': 'AVG',
    'babip': 'BABIP', 'spd': 'Spd', 'fip': 'FIP', 'whip': 'WHIP', 'wpct': 'WPCT',
    'k/9': 'K/9', 'bb/9': 'BB/9', 'k/bb': 'K/BB', 'hr/9': 'HR/9', 'era': 'ERA',
    'k%': 'K%', 'bb%': 'BB%', 'k-bb%': 'K-BB%', 'lob%': 'LOB%', 'e-f': 'ERA-FIP',
}
_TOK = {'conf': 'Conf.', 'wp': 'WP', 'rpi': 'RPI', 'sos': 'SOS', 'pe': 'PE',
        'nc': 'NC', 'wpct': 'WPCT', 'rec': 'Rec', 'pct': '%',
        'q1': 'Q1', 'q2': 'Q2', 'q3': 'Q3', 'q4': 'Q4'}

def _fallback(base):
    out = []
    for p in base.split('_'):
        pl = p.lower()
        if pl in _TOK:                    out.append(_TOK[pl])
        elif len(p) <= 3 and p.isalpha(): out.append(p.upper())
        else:                             out.append(p.capitalize())
    return ' '.join(out)

def _pretty(col):
    if col == 'age':          return 'Age'
    if col == 'role':         return 'Role'
    if col == 'is_post10':    return 'Round 11+'
    for suf in ('_bat', '_pitch', '_team'):
        if col.endswith(suf):
            base, grp = col[:-len(suf)], suf[1:]
            disp = BASE_OVERRIDES.get(base.lower()) or _fallback(base)
            return f"{disp} ({grp})"
    if col.startswith('api_'):
        return 'API ' + _fallback(col[4:])
    return BASE_OVERRIDES.get(col.lower()) or _fallback(col)

def save_fig(fig, stage, name, dpi=None, pdf_only=False):
    """Write figures/s{stage}_{name}.{png,pdf}. stage=None -> no prefix."""
    if not SAVE_FIGS:
        return
    os.makedirs(FIG_DIR, exist_ok=True)
    stem = f"{FIG_DIR}/{'' if stage is None else f's{stage}_'}{name}"
    if not pdf_only:
        fig.savefig(f'{stem}.png', dpi=dpi or FIG_DPI, bbox_inches='tight')
    fig.savefig(f'{stem}.pdf', bbox_inches='tight')
    print(f"  saved {stem}.pdf")

def gain_table(model, label='', top=200, show=True):
    """Gain-importance table, sorted and normalized to % of total gain."""
    gain = model.get_booster().get_score(importance_type='gain')
    t = (pd.DataFrame({'Feature': list(gain.keys()), 'Importance': list(gain.values())})
         .sort_values('Importance', ascending=False).reset_index(drop=True))
    t.insert(0, 'Rank', '#' + (t.index + 1).astype(str))
    t['Label'] = t['Feature'].map(_pretty)
    t['% of gain'] = (100 * t['Importance'] / t['Importance'].sum()).round(2)
    if show:
        print(f"\nFeatures by gain{f' - {label}' if label else ''} "
              f"({len(t)} used of {model.n_features_in_}):")
        with pd.option_context('display.max_rows', None):
            print(t.head(top).to_string(index=False))
    return t

def spearman_ci(a, b, alpha=0.05):
    """Spearman rho with a Fisher z-transform confidence interval."""
    rho, p = spearmanr(a, b)
    n = len(a)
    z, se, zc = np.arctanh(rho), 1 / np.sqrt(max(n - 3, 1)), 1.96
    lo, hi = np.tanh(z - zc * se), np.tanh(z + zc * se)
    print(f"  Spearman: {rho:.3f} (p={p:.2e})  95% CI [{lo:.3f}, {hi:.3f}]  n={n}")
    return rho, p, lo, hi

def _binary_cols(X):
    return {c for c in X.columns if X[c].dropna().nunique() <= 2}

def _top_feature_idx(model, X, top_n, skip_binary=False):
    """Indices of the top-n features by importance, positionally into X."""
    imp = model.feature_importances_
    order = np.argsort(imp)[::-1]
    skip = _binary_cols(X) if skip_binary else set()
    out = [int(i) for i in order if X.columns[i] not in skip and imp[i] > 0]
    return out[:top_n]

def plot_feature_importance(model, stage, top_n=10, as_percent=True, n_highlight=1,
                            name='feature_importance', title=None):
    gain = model.get_booster().get_score(importance_type='gain')
    if not gain:
        print('  (model used no features)'); return
    total = sum(gain.values()) if as_percent else 1.0
    items = sorted(gain.items(), key=lambda kv: kv[1], reverse=True)[:top_n]
    labels = [_pretty(k) for k, _ in items][::-1]
    values = [(v / total * 100.0) if as_percent else v for _, v in items][::-1]
    colors = [NAVY] * len(values)
    for j in range(min(n_highlight, len(colors))):
        colors[-(j + 1)] = RED

    fig, ax = plt.subplots(figsize=(11, 4.6))
    ypos = range(len(values))
    ax.barh(ypos, values, color=colors, height=0.62, zorder=3)
    vmax = max(values)
    for yi, v in zip(ypos, values):
        ax.text(v + vmax * 0.012, yi, f"{v:.2f}", va='center', ha='left',
                fontsize=13, color='#222222')
    ax.set_yticks(list(ypos)); ax.set_yticklabels(labels, fontsize=13)
    ax.set_xlim(0, vmax * 1.12)
    ax.set_xlabel('% of total gain' if as_percent else 'gain', fontsize=13)
    ax.set_title(title or f'{STAGE_TITLE[stage]} - Top {len(values)} Features (Gain)',
                 fontsize=17, color=NAVY, pad=16)
    ax.xaxis.grid(True, color=GRAY, linewidth=0.9, zorder=0); ax.set_axisbelow(True)
    for s in ('top', 'right', 'left'):
        ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_color('#bbbbbb')
    ax.tick_params(left=False); ax.tick_params(axis='x', labelsize=12, colors='#444444')
    plt.tight_layout()
    save_fig(fig, stage, name)
    plt.show()

def plot_pdp_grid(model, X, stage, top_n=6, name='pdp', skip_binary=True, title=None):
    """1D partial dependence for the top features. Binary columns are skipped by default,
    since a two-point line reads as a rendering bug."""
    idx = _top_feature_idx(model, X, top_n, skip_binary=skip_binary)
    if not idx:
        print('  (no non-binary features to plot)'); return
    Xp = X.fillna(X.median(numeric_only=True))
    ncol = 3
    nrow = int(np.ceil(len(idx) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.3 * ncol, 4 * nrow), squeeze=False)
    PartialDependenceDisplay.from_estimator(
        model, Xp, features=idx, kind='average',
        ax=axes.flatten()[:len(idx)], grid_resolution=50)
    for ax in axes.flatten()[len(idx):]:
        ax.axis('off')
    for ax in axes[:, 1:].flatten():
        ax.set_ylabel('')
    fig.suptitle(title or f'{STAGE_TITLE[stage]} - Partial Dependence', fontsize=14)
    plt.tight_layout()
    save_fig(fig, stage, name)
    plt.show()

def plot_pdp_2d(model, X, stage, pairs=None, top_n=4, name='pdp_2d', title=None):
    """2D interaction PDPs. Defaults to pairing the top features (0,1), (0,2), (1,3)."""
    if pairs is None:
        idx = _top_feature_idx(model, X, top_n, skip_binary=True)
        if len(idx) < 4:
            print('  (not enough features for interaction pairs)'); return
        pairs = [(idx[0], idx[1]), (idx[0], idx[2]), (idx[1], idx[3])]
    else:
        pairs = [tuple(X.columns.get_loc(c) if isinstance(c, str) else c for c in p) for p in pairs]
    Xp = X.fillna(X.median(numeric_only=True)).astype(float)
    fig, axes = plt.subplots(1, len(pairs), figsize=(6.7 * len(pairs), 5))
    axes = np.atleast_1d(axes)
    for ax, pair in zip(axes, pairs):
        PartialDependenceDisplay.from_estimator(
            model, Xp, features=[pair], kind='average', ax=ax, grid_resolution=30)
        ax.set_xlabel(_pretty(Xp.columns[pair[0]])); ax.set_ylabel(_pretty(Xp.columns[pair[1]]))
    fig.suptitle(title or f'{STAGE_TITLE[stage]} - Feature Interactions', fontsize=14)
    plt.tight_layout()
    save_fig(fig, stage, name)
    plt.show()

def plot_shap_summary(model, X, stage, sample=1500, name='shap_summary'):
    """SHAP beeswarm for the top features."""
    try:
        Xs = X.sample(min(sample, len(X)), random_state=42)
        sv = shap.TreeExplainer(model).shap_values(Xs)
        if isinstance(sv, list):
            sv = sv[1]
        fig = plt.figure()
        shap.summary_plot(sv, Xs, feature_names=[_pretty(c) for c in Xs.columns],
                          max_display=12, show=False, plot_size=(10, 6))
        plt.title(f'{STAGE_TITLE[stage]} - SHAP', fontsize=13)
        plt.tight_layout()
        save_fig(plt.gcf(), stage, name)
        plt.show()
    except Exception as e:
        print(f"  SHAP summary skipped ({e.__class__.__name__}: {e})")

def plot_pcp(frame, cols, color_col, stage, name, title=None, sample=4000,
             scale=None, reverse=False):
    """Plotly parallel coordinates. PDF only, since each export is several MB."""
    d = frame[cols + [color_col]].dropna()
    if len(d) > sample:
        d = d.sample(sample, random_state=42)
    fig = px.parallel_coordinates(
        d, dimensions=cols, color=color_col,
        labels={c: _pretty(c) for c in cols},
        color_continuous_scale=scale or ['#EF553B', '#636EFA'],
        title=title or f'{STAGE_TITLE[stage]} - {_pretty(color_col)}')
    fig.update_layout(width=1000, height=500)
    if reverse:
        fig.update_coloraxes(reversescale=True)
    if SAVE_FIGS:
        os.makedirs(FIG_DIR, exist_ok=True)
        try:
            fig.write_image(f'{FIG_DIR}/s{stage}_{name}.pdf', scale=2)
            print(f"  saved {FIG_DIR}/s{stage}_{name}.pdf")
        except Exception as e:
            print(f"  PCP export skipped ({e.__class__.__name__}) -- needs kaleido")
    fig.show()

def plot_pr_curve(y_true, y_score, stage=1, name='pr_curve', label='classifier'):
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    base = float(np.mean(y_true))
    plt.rcParams.update({'font.size': 13, 'axes.labelsize': 14})
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.plot(rec, prec, lw=2.2, color='#1f77b4', label=f'{label} (PR-AUC = {ap:.3f})')
    ax.axhline(base, ls='--', lw=1.2, color='gray', label=f'Random baseline ({base:.3f})')
    ax.set_xlabel('Recall (drafted)'); ax.set_ylabel('Precision (drafted)')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3); ax.legend(loc='upper right')
    plt.tight_layout()
    save_fig(fig, stage, name, dpi=300)
    plt.show()
    return ap

def plot_calibration(y_true, y_prob, stage=1, name='calibration', bins=10):
    frac, mean_pred = calibration_curve(y_true, y_prob, n_bins=bins, strategy='quantile')
    fig, ax = plt.subplots(figsize=(5.6, 5.2))
    ax.plot([0, 1], [0, 1], 'r--', lw=1.2, label='perfect calibration')
    ax.plot(mean_pred, frac, 'o-', color=NAVY, lw=2, label='model')
    ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed drafted rate')
    ax.set_title(f'{STAGE_TITLE[stage]} - Calibration', fontsize=13)
    ax.grid(ls=':', lw=0.5, alpha=0.6); ax.legend(loc='upper left')
    plt.tight_layout()
    save_fig(fig, stage, name)
    plt.show()

def plot_score_hist(scores, stage=1, name='prob_hist', bins=50, xlabel='Predicted probability'):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(scores, bins=bins, color=NAVY, alpha=0.85)
    ax.set_xlabel(xlabel); ax.set_ylabel('Players')
    ax.set_title(f'{STAGE_TITLE[stage]} - Score Distribution', fontsize=13)
    ax.grid(axis='y', ls=':', lw=0.5, alpha=0.6)
    plt.tight_layout()
    save_fig(fig, stage, name)
    plt.show()

def plot_pred_vs_actual(actual, pred, stage, name='pred_vs_actual', xlabel='Actual',
                        ylabel='Predicted', log=False, ref=None, title=None):
    a, p = np.asarray(actual, float), np.asarray(pred, float)
    plt.rcParams.update({'font.size': 13, 'axes.labelsize': 14})
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(a, p, alpha=0.55, s=26, color=NAVY, edgecolor='none')
    lo, hi = float(min(a.min(), p.min())), float(max(a.max(), p.max()))
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.3, label='perfect')
    if ref is not None:
        ax.axhline(ref, color='gray', ls=':', lw=1); ax.axvline(ref, color='gray', ls=':', lw=1)
    if log:
        ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title, fontsize=13)
    ax.grid(ls=':', lw=0.5, alpha=0.5); ax.legend(loc='upper left')
    plt.tight_layout()
    save_fig(fig, stage, name, dpi=300)
    plt.show()

def plot_residuals(actual, pred, stage, name='residuals', hue=None, hue_label='',
                   xlabel='Predicted', ref=0.0):
    a, p = np.asarray(actual, float), np.asarray(pred, float)
    resid = p - a
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    if hue is None:
        axes[0].scatter(p, resid, alpha=0.55, s=26, color=NAVY, edgecolor='none')
    else:
        h = np.asarray(hue)
        for val, col in zip(np.unique(h), [NAVY, RED, '#2a9d8f', '#e9c46a']):
            m = h == val
            axes[0].scatter(p[m], resid[m], alpha=0.6, s=26, color=col, edgecolor='none',
                            label=f'{hue_label}={val}')
        axes[0].legend(fontsize=10)
    axes[0].axhline(ref, color='r', ls='--', lw=1.3)
    axes[0].set_xlabel(xlabel); axes[0].set_ylabel('Residual (pred - actual)')
    axes[0].grid(ls=':', lw=0.5, alpha=0.5)
    axes[1].hist(resid, bins=40, color=NAVY, alpha=0.85)
    axes[1].axvline(ref, color='r', ls='--', lw=1.3)
    axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Players')
    axes[1].grid(axis='y', ls=':', lw=0.5, alpha=0.6)
    fig.suptitle(f'{STAGE_TITLE[stage]} - Residuals', fontsize=14)
    plt.tight_layout()
    save_fig(fig, stage, name)
    plt.show()

def plot_target_dist(series, stage, name='target_dist', by=None, by_label='', ref=None,
                     logx=False, year=None):
    s = pd.Series(series).dropna()
    ncol = 2 if year is None else 3
    fig, axes = plt.subplots(1, ncol, figsize=(6 * ncol, 4.6))
    axes[0].hist(s, bins=60, color=NAVY, alpha=0.85)
    if ref is not None:
        axes[0].axvline(ref, color=RED, ls='--', lw=1.6, label=f'parity ({ref})')
        axes[0].legend(fontsize=10)
    if logx:
        axes[0].set_xscale('log')
    axes[0].set_xlabel(_pretty(name)); axes[0].set_ylabel('Players')
    axes[0].set_title('Distribution', fontsize=12)
    axes[0].grid(axis='y', ls=':', lw=0.5, alpha=0.6)
    if by is not None:
        groups = [s.values[np.asarray(by) == v] for v in np.unique(by)]
        axes[1].boxplot(groups, tick_labels=[f'{by_label}={v}' for v in np.unique(by)])
        axes[1].set_title(f'By {by_label or "group"}', fontsize=12)
    if ref is not None:
        axes[1].axhline(ref, color=RED, ls='--', lw=1.2)
    axes[1].grid(axis='y', ls=':', lw=0.5, alpha=0.6)
    if year is not None:
        axes[2].boxplot([s.values[np.asarray(year) == v] for v in np.unique(year)],
                        tick_labels=[str(int(v)) for v in np.unique(year)])
        if ref is not None:
            axes[2].axhline(ref, color=RED, ls='--', lw=1.2)
        axes[2].set_title('By year', fontsize=12)
        axes[2].grid(axis='y', ls=':', lw=0.5, alpha=0.6)
    fig.suptitle(f'{STAGE_TITLE[stage]} - Target', fontsize=14)
    plt.tight_layout()
    save_fig(fig, stage, name)
    plt.show()

print('figure helpers ready:',
      'importance, pdp_grid, pdp_2d, shap_summary, pcp, pr_curve, calibration,',
      'score_hist, pred_vs_actual, residuals, target_dist')

---
## Stage 1 - Draftable? classifier

Predicts `Drafted?` over the eligibility-filtered population, training on every year before 2026
and testing on 2026.

Two choices matter. The split is temporal rather than random, so no player's seasons land on both
sides of it. And every row is used: undersampling the majority class to 1:1 discards about 87% of
the negatives, which costs both accuracy and calibration. `BALANCE_MODE` still offers
`'undersample'` and `'scale_pos_weight'` for comparison.

Metrics are averaged over five seeds. A balanced holdout is a poor read on a population that is
11.7% positive, so the operational cell further down carries the numbers worth quoting.

In [ ]:
# Separate the two classes
drafted_indices     = df[df[target_drafted] == 1].index
not_drafted_indices = df[df[target_drafted] == 0].index
print(f"class distribution: not drafted {len(not_drafted_indices)}, drafted {len(drafted_indices)} "
      f"({len(drafted_indices)/len(df):.2%} positive)")

# Undersampling throws away ~87% of the negatives, so the default keeps every row
if BALANCE_MODE == 'undersample':
    np.random.seed(40)
    undersampled_not_drafted = np.random.choice(
        not_drafted_indices, size=len(drafted_indices), replace=False)
    train_pool = np.concatenate([drafted_indices.values, undersampled_not_drafted])
    np.random.shuffle(train_pool)
else:
    undersampled_not_drafted = np.array([], dtype=not_drafted_indices.dtype)
    train_pool = df.index.values

X_pool, y_pool = X.loc[train_pool], y.loc[train_pool]
yr_pool = df.loc[train_pool, year_col]
print(f"training pool: {len(y_pool)} rows, {y_pool.mean():.1%} positive (mode={BALANCE_MODE})")

# Hold out the most recent year rather than a random slice, so no player appears on both sides
if SPLIT_MODE == 'temporal':
    _tr = yr_pool < TEST_YEAR
    X_train, X_test = X_pool[_tr], X_pool[~_tr]
    y_train, y_test = y_pool[_tr], y_pool[~_tr]
    print(f"temporal split: train years {sorted(yr_pool[_tr].unique())} n={len(y_train)} | "
          f"test {TEST_YEAR} n={len(y_test)}")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X_pool, y_pool, test_size=0.2, random_state=42, stratify=y_pool)
    print(f"random split: train n={len(y_train)} | test n={len(y_test)}")

neg_count, pos_count = int((y_train == 0).sum()), int((y_train == 1).sum())
scale_pos_weight = neg_count / max(pos_count, 1)
print(f"train class 0={neg_count} class 1={pos_count} scale_pos_weight={scale_pos_weight:.2f}")

In [ ]:
# Train XGBoost classifier, averaging the reported metrics over several seeds
_clf_kw = dict(RP_CLF)
if BALANCE_MODE == 'scale_pos_weight':
    _clf_kw['scale_pos_weight'] = scale_pos_weight

_runs = []
for _seed in SEEDS:
    _m = xgb.XGBClassifier(**dict(_clf_kw, random_state=_seed))
    _m.fit(X_train, y_train)
    _p = _m.predict_proba(X_test)[:, 1]
    _runs.append({'seed': _seed,
                  'accuracy': accuracy_score(y_test, (_p >= 0.5).astype(int)),
                  'roc_auc': roc_auc_score(y_test, _p),
                  'pr_auc': average_precision_score(y_test, _p),
                  'brier': brier_score_loss(y_test, _p)})

# Keep the first seed's model as the one every later cell explains and plots
draft_model = xgb.XGBClassifier(**dict(_clf_kw, random_state=SEEDS[0]))
draft_model.fit(X_train, y_train)
y_proba = draft_model.predict_proba(X_test)[:, 1]
y_pred = draft_model.predict(X_test)

stage1_seeds = pd.DataFrame(_runs).set_index('seed')
print(f"\nStage 1 ({SPLIT_MODE} split, {BALANCE_MODE}), test year {TEST_YEAR}, "
      f"{len(SEEDS)} seeds:")
display(stage1_seeds.round(4))
print(stage1_seeds.agg(['mean', 'std']).round(4).to_string())
print()
print(classification_report(y_test, y_pred, digits=3))

In [ ]:
# Show top features by importance from the trained classifier
_ = gain_table(draft_model, label='Stage 1 classifier')

In [ ]:
# Top-10 feature importance bar chart for the paper
plot_feature_importance(draft_model, stage=1)

In [ ]:
# SHAP summary, to check the importance ranking against per-player attributions
plot_shap_summary(draft_model, X_train, stage=1)

In [ ]:
# Pick the features on the PCP axes
pcp_features = ['rpi_team', 'era_pitch', 'whip_pitch', 'k/9_pitch', 'bb/9_pitch',
                'so_pitch', 'ip_pitch']

plot_pcp(df, pcp_features, target_drafted, stage=1, name='pcp_actual',
         title='Stage 1 - Pitching profile by actual draft outcome')

# Same axes coloured by what the model predicted, so the two can be compared by eye
_pcp_pred = X_test[pcp_features].copy()
_pcp_pred['predicted'] = y_pred
plot_pcp(_pcp_pred, pcp_features, 'predicted', stage=1, name='pcp_predicted',
         title='Stage 1 - Pitching profile by predicted class')

_pcp_prob = X_test[pcp_features].copy()
_pcp_prob['draft_probability'] = y_proba
plot_pcp(_pcp_prob, pcp_features, 'draft_probability', stage=1, name='pcp_probability',
         title='Stage 1 - Pitching profile by predicted probability')

In [ ]:
# Spread of predicted draft probabilities on the held-out year
plot_score_hist(y_proba, stage=1)

In [ ]:
# Do the predicted probabilities match observed draft rates?
plot_calibration(y_test, y_proba, stage=1)

In [ ]:
# Partial dependence for the top features - how each one moves draft probability
plot_pdp_grid(draft_model, X_train, stage=1)

In [ ]:
# Two-feature interactions worth showing in the paper
plot_pdp_2d(draft_model, X_train, stage=1,
            pairs=[('age', 'so_pitch'), ('age', 'wrc_bat'), ('rpi_team', 'wrc_bat')])

In [ ]:
# PDP next to SHAP dependence for the same features
# AI CREATE SOME SHAP DEPENDENCE PLOTS

# Top features shared by both columns
top_n = 3
importances = draft_model.feature_importances_
top_indices = np.argsort(importances)[::-1][:top_n].tolist()
top_names = X_train.columns[top_indices].tolist()

# PDP needs a NaN-free grid; SHAP can use the real (NaN-containing) data
X_pdp = X_train.fillna(X_train.median())

# Compute SHAP once, reuse for every right-column panel (TreeExplainer is exact + fast for XGB)
explainer = shap.TreeExplainer(draft_model)
shap_exp = explainer.shap_values(X_train)
if isinstance(shap_exp, list):      # older SHAP returns [class0, class1]
    shap_exp = shap_exp[1]          # keep positive (drafted) class

fig, axes = plt.subplots(top_n, 2, figsize=(12, 4 * top_n))

for row, (idx, name) in enumerate(zip(top_indices, top_names)):
    # Left: partial dependence
    PartialDependenceDisplay.from_estimator(
        draft_model, X_pdp, features=[idx],
        kind='average', ax=axes[row, 0], grid_resolution=50
    )
    axes[row, 0].set_title(f'PDP — {name}', fontsize=11)

    # Right: SHAP dependence on real data, auto-colored by top interaction
    shap.dependence_plot(
        idx, shap_exp, X_train,
        ax=axes[row, 1], show=False, interaction_index='auto'
    )
    axes[row, 1].set_title(f'SHAP dependence — {name}', fontsize=11)

fig.suptitle('Draft Probability — PDP (left) vs SHAP dependence (right)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# USED AI TO MAKE GRAPHIC FOR PAPER

# Get top features dynamically (for 1D PDPs only)
top_indices = _top_feature_idx(draft_model, X_train, 6, skip_binary=True)
top_names = [X_train.columns[i] for i in top_indices]
print("Top 6 features for PDP:", top_names)  # Sanity check

# Always use these exact interaction concepts for bottom-row 2D PDPs
cols = X_train.columns.tolist()

def find_best(cols, pattern_groups):
    for pats in pattern_groups:
        for c in cols:
            low = c.lower()
            if all(p.lower() in low for p in pats):
                return c
    return None

age_col = 'age'
so_col  = 'so_pitch'
wrc_col = 'wrc_bat'
rpi_col = 'rpi_team'

missing = [
    name for name, found in [
        ('age', age_col),
        ('strikeouts', so_col),
        ('wRC', wrc_col),
        ('team RPI', rpi_col),
    ] if not found
]
if missing:
    raise ValueError(f"Missing required columns for fixed 2D PDPs: {missing}")

# Fixed bottom-row pairs (independent of feature importance ordering)
pairs = [
    (age_col, so_col),    # age x strikeouts
    (age_col, wrc_col),   # age x wRC
    (rpi_col, wrc_col),   # team RPI x wRC
]

X_pdp = X_train.fillna(X_train.median()).astype(float)

# Single combined figure: 2 rows 1D, 1 row 2D
fig, axes = plt.subplots(3, 3, figsize=(14, 12))

# Top two rows: 1D PDPs for top 6 features
PartialDependenceDisplay.from_estimator(
    draft_model, X_pdp,
    features=top_indices,
    kind='average',
    ax=axes[:2].flatten(),
    grid_resolution=50
)

# Bottom row: fixed 2D interaction PDPs
for ax, pair in zip(axes[2], pairs):
    PartialDependenceDisplay.from_estimator(
        draft_model, X_pdp,
        features=[pair],
        kind='average',
        ax=ax,
        grid_resolution=30
    )

# Keep "Partial dependence" only on the left 1D column (top two rows).
# Bottom-row y-labels are feature names, so leave them alone.
for ax in axes[:2, 1:].flatten():
    ax.set_ylabel('')

plt.tight_layout()
plt.subplots_adjust(hspace=0.3)

row2_bottom = axes[1, 0].get_position().y0
row3_top    = axes[2, 0].get_position().y1
divider_y   = (row2_bottom + row3_top) / 2

save_fig(fig, 1, 'pdp_paper')
# s1_pdp.{pdf,png} kept under its original name too -- the paper already cites it.
save_fig(fig, 1, 'pdp')
plt.show()

In [ ]:
# Used AI to help make the PR-AUC and ROC-AUC numbers for the paper

# Operational (imbalanced) evaluation for Stage 1
#
# Reuses the trained model (no retraining) and evaluates it
# on a constructed test set at the natural 8.6% positive rate.
# This gives the reviewer-facing operational metrics (PR-AUC,
# threshold-specific precision/recall) without changing any of
# the balanced numbers

# Step 1: Build the operational test set
# Training on every row leaves the whole held-out year unseen, so the operational test set is
# just that year at its own positive rate -- nothing needs subsampling to construct it.
if SPLIT_MODE == 'temporal' and BALANCE_MODE != 'undersample':
    op_test_idx = df.index[df[year_col] == TEST_YEAR].values
else:
    # Undersampling leaves most negatives out of the pool, so rebuild the natural rate from the
    # negatives the model never saw plus the positives already held out
    unseen_not_drafted_idx = np.setdiff1d(not_drafted_indices.values, undersampled_not_drafted)
    if SPLIT_MODE == 'temporal':
        _ty_neg = df.index[(df[year_col] == TEST_YEAR) & (df[target_drafted] == 0)].values
        unseen_not_drafted_idx = np.intersect1d(unseen_not_drafted_idx, _ty_neg)
    drafted_test_idx = y_test[y_test == 1].index.values
    target_positive_rate = drafted_indices.shape[0] / df.shape[0]
    n_pos = len(drafted_test_idx)
    n_neg = min(int(round(n_pos * (1 - target_positive_rate) / target_positive_rate)),
                len(unseen_not_drafted_idx))
    np.random.seed(42)
    op_neg_idx = np.random.choice(unseen_not_drafted_idx, size=n_neg, replace=False)
    op_test_idx = np.concatenate([drafted_test_idx, op_neg_idx])

X_op_test = X.loc[op_test_idx]
y_op_test = y.loc[op_test_idx]

print(f"Operational test set:")
print(f"  Total samples: {len(y_op_test)}")
print(f"  Positives:     {y_op_test.sum()} ({y_op_test.mean()*100:.2f}%)")

# An empty negative pool would silently give PR-AUC 1.000 and ROC-AUC nan
assert y_op_test.sum() > 0 and (y_op_test == 0).sum() > 0, "operational test set is single-class"

# Step 3: Score with the existing draft_model
y_op_proba = draft_model.predict_proba(X_op_test)[:, 1]
baseline_pr = y_op_test.mean()

pr_auc  = average_precision_score(y_op_test, y_op_proba)
roc_auc = roc_auc_score(y_op_test, y_op_proba)

def metrics_at_thresh(threshold):
    y_pred = (y_op_proba >= threshold).astype(int)
    return {
        "threshold": threshold,
        "precision": precision_score(y_op_test, y_pred, zero_division=0),
        "recall":    recall_score(y_op_test, y_pred, zero_division=0),
        "f1":        f1_score(y_op_test, y_pred, zero_division=0),
        "tn_fp_fn_tp": confusion_matrix(y_op_test, y_pred).ravel().tolist(),
    }

results = [metrics_at_thresh(t) for t in (0.50, 0.40, 0.30, 0.20)]

# Step 4: Print everything cleanly
print("\n" + "=" * 64)
print("OPERATIONAL METRICS (imbalanced test set)")
print("=" * 64)
print(f"  PR-AUC:        {pr_auc:.3f}    (random baseline {baseline_pr:.3f})")
print(f"  ROC-AUC:       {roc_auc:.3f}")
print()
print(f"  {'Thresh':<8}{'Prec':>8}{'Recall':>10}{'F1':>8}    {'TN':>5} {'FP':>5} {'FN':>5} {'TP':>5}")
for r in results:
    tn, fp, fn, tp = r["tn_fp_fn_tp"]
    print(f"  {r['threshold']:<8.2f}{r['precision']:>8.3f}{r['recall']:>10.3f}"
          f"{r['f1']:>8.3f}    {tn:>5} {fp:>5} {fn:>5} {tp:>5}")

print("\nFull classification report at threshold 0.5:")
print(classification_report(
    y_op_test, (y_op_proba >= 0.5).astype(int),
    target_names=["Not drafted", "Drafted"], digits=3
))

# Step 5: Precision-recall curve + calibration on the operational distribution
plot_pr_curve(y_op_test, y_op_proba, stage=1, name='pr_curve', label='Stage 1 classifier')
plot_calibration(y_op_test, y_op_proba, stage=1, name='calibration_operational')

# Step 6: Snippet for the paper placeholders
r05 = results[0]   # threshold 0.5
r03 = results[2]   # threshold 0.3

print("\n" + "=" * 64)
print("VALUES FOR THE PAPER")
print("=" * 64)
print(f"""
ABSTRACT and CONCLUSION:
  PR-AUC {pr_auc:.3f} on the operational imbalanced distribution

RESULTS PARAGRAPH (Stage 1, operational):
  PR-AUC:                       {pr_auc:.3f}
  ROC-AUC:                      {roc_auc:.3f}
  Random baseline (PR):         {baseline_pr:.3f}
  At threshold 0.5: precision = {r05['precision']:.3f}, recall = {r05['recall']:.3f}
  At threshold 0.3: precision = {r03['precision']:.3f}, recall = {r03['recall']:.3f}

The model achieves PR-AUC of {pr_auc:.3f} on this imbalanced test set
(random baseline {baseline_pr:.3f}) and ROC-AUC of {roc_auc:.3f}.
At the default 0.5 probability threshold, operational precision on the
drafted class is {r05['precision']:.3f} with recall {r05['recall']:.3f};
lowering the threshold to 0.3 raises recall to {r03['recall']:.3f} at
the cost of precision {r03['precision']:.3f}.
""")

In [ ]:
# Balanced test set: same held-out year, negatives randomly downsampled to match the positives.
# Averaged over several draws, since a single draw of 347 from 1,911 is noisy.
#
# Precision, F1 and PR-AUC all depend on the class ratio, so this answers a different question from
# the cell above -- "can it separate one drafted player from one undrafted player" rather than
# "what happens across the whole eligible population". Recall and ROC-AUC do not move.

BALANCED_TEST_DRAWS = 5

_pos_idx = X_op_test.index[y_op_test == 1]
_neg_idx = X_op_test.index[y_op_test == 0]
print(f"held-out year: {len(_pos_idx)} drafted, {len(_neg_idx)} not drafted")

_bal_rows, _first = [], None
for _d in range(BALANCED_TEST_DRAWS):
    _rng = np.random.RandomState(100 + _d)
    _sel = np.concatenate([_pos_idx.values,
                           _rng.choice(_neg_idx.values, size=len(_pos_idx), replace=False)])
    _yb = y.loc[_sel]
    _pb = draft_model.predict_proba(X.loc[_sel])[:, 1]
    _hb = (_pb >= 0.5).astype(int)
    _bal_rows.append({'draw': _d, 'accuracy': accuracy_score(_yb, _hb),
                      'precision': precision_score(_yb, _hb, zero_division=0),
                      'recall': recall_score(_yb, _hb, zero_division=0),
                      'f1': f1_score(_yb, _hb, zero_division=0),
                      'roc_auc': roc_auc_score(_yb, _pb),
                      'pr_auc': average_precision_score(_yb, _pb)})
    if _first is None:
        _first = (_yb, _hb)

balanced_results = pd.DataFrame(_bal_rows).set_index('draw')
print(f"\nBalanced test set ({len(_pos_idx)} per class, {BALANCED_TEST_DRAWS} draws):")
display(balanced_results.round(4))
print(balanced_results.agg(['mean', 'std']).round(4).to_string())

print("\nClassification report, first draw:")
print(classification_report(_first[0], _first[1],
                            target_names=['Not drafted', 'Drafted'], digits=3))

In [ ]:
# Precision depends on how many undrafted players you screen, so quote it against a stated base
# rate. Recall and ROC-AUC are unchanged throughout -- only precision, F1 and PR-AUC move.

_rates = []
for _target_rate in [None, 0.50, 0.35, 0.25, 0.15]:
    if _target_rate is None:
        _sel, _lab = X_op_test.index.values, 'full eligible population'
    else:
        _n_neg = int(round(len(_pos_idx) * (1 - _target_rate) / _target_rate))
        if _n_neg > len(_neg_idx):
            continue
        _rng = np.random.RandomState(7)
        _sel = np.concatenate([_pos_idx.values,
                               _rng.choice(_neg_idx.values, size=_n_neg, replace=False)])
        _lab = f'shortlist at {_target_rate:.0%} drafted'
    _yb, _pb = y.loc[_sel], draft_model.predict_proba(X.loc[_sel])[:, 1]
    _hb = (_pb >= 0.5).astype(int)
    _rates.append({'population': _lab, 'n': len(_sel), 'base_rate': float(_yb.mean()),
                   'precision': precision_score(_yb, _hb, zero_division=0),
                   'recall': recall_score(_yb, _hb, zero_division=0),
                   'PR_AUC': average_precision_score(_yb, _pb),
                   'ROC_AUC': roc_auc_score(_yb, _pb)})
print('Same model and same predictions at every row; only the screening population changes:')
display(pd.DataFrame(_rates).set_index('population').round(4))

In [ ]:
# Isolate the two changes that landed together: the eligibility filter, and the switch from a
# random split to a temporal one. All three rows use the older balanced-holdout protocol so the
# numbers are comparable; V10's own headline uses a different test distribution (see below).

_legacy = {}
for _tag, _frame, _temporal in [
        ('random split, unfiltered',            df_all, False),
        ('random split, eligibility-filtered',  df,     False),
        ('temporal split, eligibility-filtered', df,    True)]:
    _Xa, _ya = _frame[features], _frame[target_drafted].astype(int)
    _pos = _frame[_frame[target_drafted] == 1].index
    _neg = _frame[_frame[target_drafted] == 0].index
    np.random.seed(40)
    _bal = np.concatenate([_pos.values, np.random.choice(_neg, size=len(_pos), replace=False)])
    np.random.shuffle(_bal)
    if _temporal:
        _yr = _frame.loc[_bal, year_col]
        _trm = (_yr < TEST_YEAR).values
        _Xtr, _Xte = _Xa.loc[_bal][_trm], _Xa.loc[_bal][~_trm]
        _ytr, _yte = _ya.loc[_bal][_trm], _ya.loc[_bal][~_trm]
    else:
        _Xtr, _Xte, _ytr, _yte = train_test_split(
            _Xa.loc[_bal], _ya.loc[_bal], test_size=0.2, random_state=42, stratify=_ya.loc[_bal])
    _m = xgb.XGBClassifier(**RP_CLF).fit(_Xtr, _ytr)
    _p = _m.predict_proba(_Xte)[:, 1]
    _legacy[_tag] = dict(n=len(_yte), accuracy=accuracy_score(_yte, (_p >= 0.5).astype(int)),
                         roc_auc=roc_auc_score(_yte, _p), pr_auc=average_precision_score(_yte, _p))

print("Balanced-holdout protocol, so rows are comparable:")
display(pd.DataFrame(_legacy).T.round(3))
print("Row 1 -> 2 isolates the eligibility filter; row 2 -> 3 isolates the split change.")
print()
print(f"V10 as shipped trains on all {len(y_train)} rows and is scored on the whole {TEST_YEAR}")
print(f"population ({len(y_test)} rows, {y_test.mean():.1%} positive), which is a harder and more")
print(f"realistic distribution -- those numbers are in the results table below, not this one.")

In [ ]:
# Stage 1 grid search, off by default. Scored by inner leave-one-year-out over the training
# years only, so the held-out year plays no part in choosing parameters.

if RUN_HPO:
    _grid = {'max_depth': [4, 6, 8], 'learning_rate': [0.03, 0.05, 0.10],
             'min_child_weight': [1, 5, 20], 'reg_lambda': [1.0, 5.0, 20.0]}
    _train_years = [y for y in sorted(df[year_col].unique()) if y < TEST_YEAR]

    def _inner_score(params):
        scores = []
        for _y in _train_years:
            _a = df[(df[year_col] != _y) & (df[year_col] < TEST_YEAR)]
            _b = df[df[year_col] == _y]
            _m = xgb.XGBClassifier(**params).fit(_a[features], _a[target_drafted])
            scores.append(average_precision_score(
                _b[target_drafted], _m.predict_proba(_b[features])[:, 1]))
        return float(np.mean(scores))

    _rows = []
    for _combo in itertools.product(*_grid.values()):
        _p = dict(RP_CLF, **dict(zip(_grid, _combo)))
        _rows.append({**dict(zip(_grid, _combo)), 'inner_pr_auc': _inner_score(_p)})
    _hpo = pd.DataFrame(_rows).sort_values('inner_pr_auc', ascending=False)
    print(f"baseline RP_CLF inner PR-AUC: {_inner_score(dict(RP_CLF)):.4f}")
    display(_hpo.head(10).round(4))
else:
    print('RUN_HPO is off. The search was run once over 81 configs and its winner')
    print('(max_depth=4, learning_rate=0.03, min_child_weight=1, reg_lambda=1.0) improved the')
    print('inner score 0.807 -> 0.815 but gained only +0.0004 PR-AUC on the held-out year while')
    print('losing 0.006 ROC-AUC, so RP_CLF was left as it is.')

In [ ]:
# Stage 1 results table - every number the paper quotes for this stage

_op_pred = (y_op_proba >= 0.5).astype(int)
stage1_results = pd.DataFrame([
    {'metric': 'PR-AUC',              'value': average_precision_score(y_op_test, y_op_proba)},
    {'metric': 'PR-AUC baseline',     'value': float(y_op_test.mean())},
    {'metric': 'ROC-AUC',             'value': roc_auc_score(y_op_test, y_op_proba)},
    {'metric': 'Brier',               'value': brier_score_loss(y_op_test, y_op_proba)},
    {'metric': 'precision @ 0.50',    'value': precision_score(y_op_test, _op_pred, zero_division=0)},
    {'metric': 'recall @ 0.50',       'value': recall_score(y_op_test, _op_pred, zero_division=0)},
    {'metric': 'n test',              'value': float(len(y_op_test))},
    {'metric': 'positive rate',       'value': float(y_op_test.mean())},
]).set_index('metric')
print(f"Stage 1, operational distribution, test year {TEST_YEAR}")
display(stage1_results.round(4))

# Leave-one-year-out on the training years, to report a spread next to the single-year number
_loyo = {}
for _y in [y for y in sorted(df[year_col].unique()) if y < TEST_YEAR]:
    _a = df[(df[year_col] != _y) & (df[year_col] < TEST_YEAR)]
    _b = df[df[year_col] == _y]
    _m = xgb.XGBClassifier(**dict(RP_CLF, random_state=SEEDS[0]))
    _m.fit(_a[features], _a[target_drafted])
    _loyo[int(_y)] = average_precision_score(
        _b[target_drafted], _m.predict_proba(_b[features])[:, 1])
print(f"leave-one-year-out PR-AUC: {_loyo}")
print(f"  mean {np.mean(list(_loyo.values())):.4f}  min {min(_loyo.values()):.4f}  "
      f"max {max(_loyo.values()):.4f}")

In [ ]:
# Show top features with missing values
# Show top features with missing values
missing_summary = X.isnull().mean().sort_values(ascending=False)
print(missing_summary[missing_summary > 0].head(15))

---
## Bridge - drafted subset, StatsAPI merge, college draft order

Two feature sets are named here and used consistently from this point on.

`pick_features` is the stats and team columns plus the `api_*` biometrics, for retrospective
analysis of players already known to be drafted, where height, weight and handedness are legitimately
observed. `sim_*_features` is the same list with `api_*` stripped, for anything pre-draft, where
those fields do not exist yet and every player has to be scorable.

Neither can be dropped: removing the biometrics costs Stage 2 accuracy, and keeping them makes the
simulation impossible.

In [ ]:
# Now to set up for pick prediction among drafted players

target_drafted = 'Drafted?'
target_pick = 'Pick'
year_col = 'year'

# Keep only drafted players
drafted_df = df[df[target_drafted] == 1].copy()
pd.options.display.max_columns = None
display(drafted_df.head(5))

In [ ]:
# USED AI TO LOAD AND MERGE MLB Stats API draft data for pick prediction

def _slot_millions(raw):
    """pickValue -> millions of dollars, or None if there is no real slot value.

    In 2025 and 2026 every round-11+ pick carries pickValue as the literal string "0", which is
    truthy, so a plain falsy check stores 0.0 instead of None. Real slot values exist only for
    rounds 1-10 plus compensation rounds; later picks get a threshold instead, applied in Stage 3.
    """
    if raw is None:
        return None
    try:
        v = float(str(raw).replace(",", "").strip())
    except (TypeError, ValueError):
        return None
    return v / 1e6 if v > 0 else None


def load_draft_jsons(json_dir: str) -> pd.DataFrame:
    """Load all mlb_draft_YYYY.json files and flatten into a DataFrame."""
    rows = []
    for fname in sorted(os.listdir(json_dir)):
        if not fname.endswith(".json"):
            continue
        print(f"  Loading {fname}...")
        with open(os.path.join(json_dir, fname), "r", encoding="utf-8") as f:
            data = json.load(f)

        draft_year = data.get("drafts", {}).get("draftYear")
        if draft_year is None:
            print(f"    ⚠️ No draftYear found, skipping")
            continue

        for rd in data.get("drafts", {}).get("rounds", []):
            for pick in rd.get("picks", []):
                person = pick.get("person", {}) or {}
                pos = person.get("primaryPosition", {}) or {}
                bat = person.get("batSide", {}) or {}
                pitch = person.get("pitchHand", {}) or {}

                ov = pick.get("pickOverall") or pick.get("displayPickNumber")

                rows.append({
                    "year": int(draft_year),
                    "Pick": float(ov) if ov else None,
                    "api_name": person.get("fullName"),
                    "api_batSide": bat.get("code"),
                    "api_pitchHand": pitch.get("code"),
                    "api_primaryPos": pos.get("abbreviation"),
                    "api_weight": person.get("weight"),
                    "api_height": person.get("height"),
                    "api_pickValue": _slot_millions(pick.get("pickValue")),
                    "api_signingBonus": float(pick["signingBonus"]) / 1e6 if pick.get("signingBonus") else None,
                })

    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠️ No picks loaded! Check your JSON_DIR path and file contents.")
        return df
    return df.drop_duplicates(subset=["year", "Pick"])


def name_similarity(a, b) -> float:
    if pd.isna(a) or pd.isna(b):
        return 0.0
    return round(SequenceMatcher(None, str(a).lower(), str(b).lower()).ratio() * 100, 1)


def height_to_inches(h):
    if pd.isna(h):
        return None
    try:
        parts = str(h).replace('"', '').split("'")
        return int(parts[0].strip()) * 12 + int(parts[1].strip())
    except (ValueError, IndexError):
        return None


# ── Load JSONs ────────────────────────────────────────────────────────
print(f"Looking in: {os.path.abspath(JSON_DIR)}")
print(f"Files found: {os.listdir(JSON_DIR)}\n")

api_df = load_draft_jsons(JSON_DIR)

print(f"\nAPI data: {len(api_df)} picks across years {sorted(api_df['year'].unique()) if not api_df.empty else 'NONE'}")
print(f"Drafted data: {len(drafted_df)} rows, Pick dtype={drafted_df['Pick'].dtype}, year dtype={drafted_df['year'].dtype}")

# ── Print unique string values before mapping ─────────────────────────
print(f"\nUnique api_batSide values:   {sorted(api_df['api_batSide'].dropna().unique())}")
print(f"Unique api_pitchHand values: {sorted(api_df['api_pitchHand'].dropna().unique())}")
print(f"Unique api_primaryPos values: {sorted(api_df['api_primaryPos'].dropna().unique())}")
print(f"Unique api_height values:    {sorted(api_df['api_height'].dropna().unique())}")

# ── Ensure matching dtypes ────────────────────────────────────────────
drafted_df["Pick"] = drafted_df["Pick"].astype(float)
drafted_df["year"] = drafted_df["year"].astype(int)

if not api_df.empty:
    api_df["Pick"] = api_df["Pick"].astype(float)
    api_df["year"] = api_df["year"].astype(int)

    # Spot check: do the keys overlap?
    draft_keys = set(zip(drafted_df["year"], drafted_df["Pick"]))
    api_keys = set(zip(api_df["year"], api_df["Pick"]))
    overlap = draft_keys & api_keys
    print(f"\nKey overlap: {len(overlap)} matches out of {len(draft_keys)} drafted rows")

    # Merge
    drafted_df = drafted_df.merge(api_df, on=["year", "Pick"], how="left")

    # Name match
    drafted_df["name_match_pct"] = drafted_df.apply(
        lambda r: name_similarity(r["nameascii"], r["api_name"]), axis=1
    )

    print(f"\nMatched: {drafted_df['api_name'].notna().sum()} / {len(drafted_df)}")

    # Drop bad matches
    MATCH_THRESHOLD = 70
    low_match = drafted_df[drafted_df["name_match_pct"].between(1, MATCH_THRESHOLD)]
    if len(low_match) > 0:
        print(f"\n⚠️ Dropping {len(low_match)} rows with name match < {MATCH_THRESHOLD}%:")
        print(low_match[["year", "Pick", "nameascii", "api_name", "name_match_pct"]].to_string(index=False))

    drafted_df = drafted_df[~drafted_df["name_match_pct"].between(1, MATCH_THRESHOLD)].copy()
    print(f"\nRows remaining: {len(drafted_df)}")

    # ── Encode string columns to numeric ──────────────────────────────
    bat_map = {"R": 0, "L": 1, "S": 2}
    drafted_df["api_batSide"] = drafted_df["api_batSide"].map(bat_map)

    pitch_map = {"R": 0, "L": 1, "S": 2}
    drafted_df["api_pitchHand"] = drafted_df["api_pitchHand"].map(pitch_map)

    pos_map = {
        "C": 1, "SS": 2, "2B": 3, "3B": 4, "CF": 5,
        "LF": 6, "RF": 7, "IF": 8, "1B": 9, "OF": 10, "DH": 11,
        "P": 12, "TWP": 13,
    }
    drafted_df["api_primaryPos"] = drafted_df["api_primaryPos"].map(pos_map)

    drafted_df["api_height"] = drafted_df["api_height"].apply(height_to_inches).astype(float)

    # Verify
    print("\nConverted dtypes:")
    for col in ["api_batSide", "api_pitchHand", "api_primaryPos", "api_height", "api_weight"]:
        print(f"  {col}: {drafted_df[col].dtype}, unique: {sorted(drafted_df[col].dropna().unique())}")

else:
    print("\n❌ No API data loaded — nothing to merge.")

In [ ]:
# Integrity of the slot partition, now that "0" is read as missing.
_dd = drafted_df
print(f"drafted rows after the name-match filter: {len(_dd)}")
print(f"  api_pickValue present: {int(_dd['api_pickValue'].notna().sum())}")
print(f"  api_pickValue == 0:    {int((_dd['api_pickValue'] == 0).sum())}  (must be 0)")
assert not (_dd['api_pickValue'] == 0).any(), 'a zero slot value survived the loader fix'

_missing_slot = _dd[_dd['api_pickValue'].isna()]
print(f"  slot missing:          {len(_missing_slot)}, rounds "
      f"{sorted(_missing_slot['Round'].dropna().unique().astype(int))}")
# Round already folds compensation rounds into their base round, so rounds 1-10 is just
# Round <= 10 and the split needs no special-casing.
assert (_missing_slot['Round'] >= 11).all(), 'a round 1-10 pick has no slot value'
assert (_dd.loc[_dd['api_pickValue'].notna(), 'Round'] <= 10).all(), \
    'a round 11+ pick has an official slot value'
print(f"  signingBonus present:  {int(_dd['api_signingBonus'].notna().sum())} "
      f"({int(_dd['api_signingBonus'].isna().sum())} did not sign)")

In [ ]:
# Create college-only rank as target for each year
# Filter to college players only and rank within each year
draftedClean_df = drafted_df[drafted_df[target_drafted] == 1].copy()

# Create college-only draft order within each year
draftedClean_df = draftedClean_df.sort_values([year_col, target_pick])
draftedClean_df['College_Draft_Order'] = draftedClean_df.groupby(year_col).cumcount() + 1

# Normalize to 0-1 so rankings are comparable across years with different class sizes
draftedClean_df['College_Draft_Pct'] = draftedClean_df.groupby(year_col)['College_Draft_Order'].transform(
    lambda x: (x - 1) / (len(x) - 1) if len(x) > 1 else 0.5
)

print("College draft order examples:")
print(draftedClean_df[[year_col, 'nameascii', target_pick, 'College_Draft_Order', 'College_Draft_Pct']].head(15))
print(f"\nPlayers per year:")
print(draftedClean_df.groupby(year_col).size())

---
## Stage 2 - college draft order

In [ ]:
# XGBoost regression to predict college draft order, using just College_Draft_Order as target (not pick number)

test_year = TEST_YEAR

train_df = draftedClean_df[draftedClean_df[year_col] < test_year]
test_df  = draftedClean_df[draftedClean_df[year_col] == test_year]

api_features = ['api_batSide', 'api_pitchHand', 'api_primaryPos', 'api_weight', 'api_height']
all_features = features + api_features

pick_features   = list(all_features)                       # retrospective: includes api_*
sim_rank_features = [f for f in pick_features if not str(f).startswith('api_')]  # pre-draft

X_train = train_df[pick_features]
y_train = train_df['College_Draft_Order']

X_test = test_df[pick_features]
y_test_order = test_df['College_Draft_Order']

# No early stopping -- there is no validation year to stop on
model_rank = xgb.XGBRegressor(**RP_REG, eval_metric='mae')
model_rank.fit(X_train, y_train)

preds = model_rank.predict(X_test)
pred_order = pd.Series(preds).rank(method='min').astype(int).values
actual_order = y_test_order.values

print(f"College draft class sizes per year:")
print(draftedClean_df.groupby(year_col).size())

corr_order, p_value = spearmanr(actual_order, pred_order)
mae_order = mean_absolute_error(actual_order, pred_order)

print(f"\nCollege-Only Rank Model — Raw Order Target (n={len(y_test_order)}):")
print(f"  Spearman: {corr_order:.3f} (p={p_value:.2e})")
print(f"  MAE:      {mae_order:.1f}")

In [ ]:
# Fisher z-transform for confidence interval
corr_order, p_value, ci_low, ci_high = spearman_ci(actual_order, pred_order)

In [ ]:
# Show top features by importance from the rank model
_ = gain_table(model_rank, label='Stage 2 rank model')

In [ ]:
# Top-10 feature importance for the rank model
plot_feature_importance(model_rank, stage=2)

In [ ]:
# SHAP summary for the rank model
plot_shap_summary(model_rank, X_train, stage=2)

In [ ]:
# Compare ranks, not raw output -- the regressor's scale is not calibrated to draft order.
# Axes are inverted so pick 1 sits top-left; the second panel zooms the top of the board.
plt.rcParams.update({'font.size': 13, 'axes.labelsize': 14})

for _name, _lim in [('rank_scatter_full', None), ('rank_scatter_zoom', 50)]:
    _m = np.ones(len(actual_order), bool) if _lim is None else (actual_order <= _lim)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(actual_order[_m], pred_order[_m], alpha=0.6, s=30, color=NAVY, edgecolor='none')
    _hi = max(actual_order[_m].max(), pred_order[_m].max())
    ax.plot([1, _hi], [1, _hi], 'r--', lw=1.3, label='perfect')
    if _lim is not None:
        _sub = np.argsort(actual_order[_m])[:20]
        for _i in _sub:
            ax.annotate(test_df['nameascii'].values[_m][_i],
                        (actual_order[_m][_i], pred_order[_m][_i]),
                        fontsize=6, alpha=0.7)
    ax.set_xlim(_hi, 1); ax.set_ylim(_hi, 1)          # inverted: pick 1 top-left
    ax.set_xlabel('Actual college draft order'); ax.set_ylabel('Predicted college draft order')
    ax.grid(ls=':', lw=0.5, alpha=0.5); ax.legend(loc='lower right')
    plt.tight_layout()
    save_fig(fig, 2, _name, dpi=300)
    plt.show()

plot_residuals(actual_order, pred_order, stage=2, xlabel='Predicted college draft order')

In [ ]:
# Pitching profile of drafted players, coloured by where they went
plot_pcp(draftedClean_df, ['era_pitch', 'fip_pitch', 'whip_pitch', 'k/9_pitch',
                          'bb/9_pitch', 'so_pitch', 'ip_pitch'],
         'College_Draft_Order', stage=2, name='pcp',
         title='Stage 2 - Pitching profile by college draft order', reverse=True)

In [ ]:
# How each top feature moves the predicted draft order
plot_pdp_grid(model_rank, train_df[pick_features], stage=2)

In [ ]:
# Two-feature interactions for the rank model
plot_pdp_2d(model_rank, train_df[pick_features], stage=2)

In [ ]:
# ERA against college draft order
plot_df = draftedClean_df[['era_pitch', 'College_Draft_Order']].dropna()

plt.figure(figsize=(8, 5))
plt.scatter(plot_df['era_pitch'], plot_df['College_Draft_Order'], alpha=0.5)
plt.xlabel('ERA')
plt.ylabel('College Draft Order')
plt.gca().invert_yaxis()  # pick 1 at top
plt.title('ERA vs College Draft Order')
plt.show()

In [ ]:
# Build results dataframe
results_rank_df = pd.DataFrame({
    'name': test_df['nameascii'].values,
    'Predicted_College_Order': pred_order,
    'Actual_College_Order': actual_order,
})

results_rank_df = results_rank_df.sort_values('Actual_College_Order').reset_index(drop=True)

print(f"Player College Draft Order Predictions (Test Year: {test_year})")
print(f"Total college draftees: {len(results_rank_df)}\n")
display(results_rank_df.head(20))

---
## Percent-Based College Draft Order Model

In [ ]:
# XGBoost regression to predict College_Draft_Pct (normalized 0-1)
y_train_pct = train_df['College_Draft_Pct']
y_test_pct = test_df['College_Draft_Pct']

model_rank_pct = xgb.XGBRegressor(**RP_REG, eval_metric='mae')
model_rank_pct.fit(X_train, y_train_pct)

preds_pct = model_rank_pct.predict(X_test)
pred_order_pct = pd.Series(preds_pct).rank(method='min').astype(int).values

corr_pct, p_pct = spearmanr(actual_order, pred_order_pct)
mae_pct = mean_absolute_error(y_test_pct.values, preds_pct)

print(f"\nCollege-Only Rank Model - Percent Target (n={len(y_test_pct)}):")
print(f"  Spearman (rank): {corr_pct:.3f} (p={p_pct:.2e})")
print(f"  MAE (pct):       {mae_pct:.4f}")
print(f"\nComparison:")
print(f"  Raw Order Model Spearman: {corr_order:.3f}")
print(f"  Pct Model Spearman:       {corr_pct:.3f}")

In [ ]:
# Same confidence interval for the percent-target model
corr_order_pct, p_value_pct, ci_low_pct, ci_high_pct = spearman_ci(actual_order, pred_order_pct)

In [ ]:
# Percent model results table
results_pct_df = pd.DataFrame({
    'name': test_df['nameascii'].values,
    'Predicted_Pct': preds_pct,
    'Actual_Pct': y_test_pct.values,
    'Predicted_Order_from_Pct': pred_order_pct,
    'Actual_College_Order': actual_order,
})
results_pct_df = results_pct_df.sort_values('Actual_College_Order').reset_index(drop=True)
display(results_pct_df.head(20))

In [ ]:
prospects = pd.read_csv(
    f'mlb_draft_prospects/mlb_{TEST_YEAR}_draft_prospects_top250.csv')

# Compare with MLB Pipeline on matched players
results_rank_df['name_clean'] = results_rank_df['name'].str.strip().str.lower()
prospects['name_clean'] = prospects['name'].str.strip().str.lower()

comp = results_rank_df.merge(prospects[['name_clean', 'rank']], on='name_clean', how='inner')

# Re-rank everything among only matched players
comp = comp.sort_values('Actual_College_Order').reset_index(drop=True)
comp['Actual_Order_Matched'] = range(1, len(comp) + 1)

comp = comp.sort_values('Predicted_College_Order').reset_index(drop=True)
comp['Pred_Order_Matched'] = range(1, len(comp) + 1)

comp = comp.sort_values('rank').reset_index(drop=True)
comp['Pipeline_Order_Matched'] = range(1, len(comp) + 1)

corr_you, p_you = spearmanr(comp['Actual_Order_Matched'], comp['Pred_Order_Matched'])
corr_pipe, p_pipe = spearmanr(comp['Actual_Order_Matched'], comp['Pipeline_Order_Matched'])
mae_you = mean_absolute_error(comp['Actual_Order_Matched'], comp['Pred_Order_Matched'])
mae_pipe = mean_absolute_error(comp['Actual_Order_Matched'], comp['Pipeline_Order_Matched'])

print(f"Head-to-Head on Matched College Players (n={len(comp)})")
print(f"{'Comparison':<45} {'Spearman':>10}    {'P-Value':>8}  {'MAE':>10}")
print("-" * 80)
print(f"{'Your Rank Model vs Actual':<45} {corr_you:>10.3f}    {p_you:.2e} {mae_you:>10.1f}")
print(f"{'MLB Pipeline vs Actual':<45} {corr_pipe:>10.3f}    {p_pipe:.2e} {mae_pipe:>10.1f}")

comp = comp[['name', 'Pipeline_Order_Matched', 'Predicted_College_Order', 'Actual_College_Order']].copy()
comp.columns = ['name', 'MLB_Pipeline_College_Order', 'Predicted_College_Order', 'Actual_College_Order']
display(comp.sort_values('Actual_College_Order').reset_index(drop=True).head(25))

## Multi-Year MLB Pipeline Benchmark (2021–2026)

The cells below extend the cell-35 single-year comparison to all five years of MLB Pipeline top-250 prospect rankings. They:

1. Load `mlb_<year>_draft_prospects_top250.csv` for 2021–2025 from `mlb_draft_prospects/`.
2. Generate out-of-fold per-year predictions from the rank model (each year predicted by a model that did NOT see it during training).
3. Run a per-year head-to-head: **Your Model** vs **MLB Pipeline** vs **Actual draft order**.
4. Run a developmental-success benchmark on 2021–2023 cohorts using `current_level` (post-draft ground truth).

`current_level` is used **only** for the eval in cell *develop-eval*. It is never merged into the training features.

In [ ]:
# Load every year of MLB Pipeline top-250 prospect CSVs (2026 included now that it exists).
PROSPECTS_DIR = 'mlb_draft_prospects'
PROSPECT_YEARS = [2021, 2022, 2023, 2024, 2025, 2026]

prospects_by_year = {}
for y in PROSPECT_YEARS:
    p = pd.read_csv(f'{PROSPECTS_DIR}/mlb_{y}_draft_prospects_top250.csv')
    p['name_clean'] = p['name'].astype(str).str.strip().str.lower()
    p['year'] = y
    prospects_by_year[y] = p

all_prospects = pd.concat(prospects_by_year.values(), ignore_index=True)
print(f"Loaded prospects for years: {sorted(prospects_by_year.keys())}")
print(f"Total prospect rows: {len(all_prospects)}")
print(f"\ncurrent_level distribution (all years pooled):")
print(all_prospects['current_level'].fillna('BLANK').value_counts().to_string())

In [ ]:
# Out-of-fold per-year rank-model predictions for 2021-2025.
# For each year y, train on (years != y) and predict on year y.
# Result: every year has predictions from a model that did NOT see that year during training.
years_all = sorted(draftedClean_df[year_col].unique())
print(f"Running leave-one-year-out CV across: {years_all}")

oof_frames = []
for y in years_all:
    train_slice = draftedClean_df[draftedClean_df[year_col] != y]
    test_slice  = draftedClean_df[draftedClean_df[year_col] == y]

    if len(test_slice) == 0:
        continue

    X_tr = train_slice[pick_features]
    y_tr = train_slice['College_Draft_Order']
    X_te = test_slice[pick_features]
    y_te = test_slice['College_Draft_Order']

    # No eval_set: in leave-one-year-out the held-out year is the thing being scored.
    m = xgb.XGBRegressor(**RP_REG, eval_metric='mae')
    m.fit(X_tr, y_tr, verbose=False)
    preds = m.predict(X_te)
    pred_order = pd.Series(preds).rank(method='min').astype(int).values

    oof_frames.append(pd.DataFrame({
        'name': test_slice['nameascii'].values,
        'year': y,
        'Predicted_College_Order': pred_order,
        'Actual_College_Order': y_te.values,
    }))

oof_results_rank_df = pd.concat(oof_frames, ignore_index=True)
oof_results_rank_df['name_clean'] = oof_results_rank_df['name'].astype(str).str.strip().str.lower()

print(f"\nOOF predictions: {len(oof_results_rank_df)} players across {oof_results_rank_df['year'].nunique()} years")
print(oof_results_rank_df.groupby('year').size().to_string())

In [ ]:
# Per-year head-to-head: Your Model vs MLB Pipeline vs Actual draft order.
# Restricted to matched players (in both the year's draftees and the top-250 list).
per_year_rows = []
all_matched = []

for y in PROSPECT_YEARS:
    yr_oof = oof_results_rank_df[oof_results_rank_df['year'] == y]
    yr_prosp = prospects_by_year[y]
    merged = yr_oof.merge(yr_prosp[['name_clean', 'rank']], on='name_clean', how='inner')

    if len(merged) < 5:
        per_year_rows.append({'year': y, 'n_matched': len(merged),
                              'spearman_model': np.nan, 'spearman_pipeline': np.nan,
                              'mae_model': np.nan, 'mae_pipeline': np.nan})
        continue

    merged = merged.sort_values('Actual_College_Order').reset_index(drop=True)
    merged['actual_order_m'] = range(1, len(merged) + 1)
    merged = merged.sort_values('Predicted_College_Order').reset_index(drop=True)
    merged['pred_order_m'] = range(1, len(merged) + 1)
    merged = merged.sort_values('rank').reset_index(drop=True)
    merged['pipeline_order_m'] = range(1, len(merged) + 1)

    sp_m, _ = spearmanr(merged['actual_order_m'], merged['pred_order_m'])
    sp_p, _ = spearmanr(merged['actual_order_m'], merged['pipeline_order_m'])
    mae_m = mean_absolute_error(merged['actual_order_m'], merged['pred_order_m'])
    mae_p = mean_absolute_error(merged['actual_order_m'], merged['pipeline_order_m'])

    per_year_rows.append({'year': y, 'n_matched': len(merged),
                          'spearman_model': sp_m, 'spearman_pipeline': sp_p,
                          'mae_model': mae_m, 'mae_pipeline': mae_p})
    merged['year'] = y
    all_matched.append(merged)

per_year_h2h_df = pd.DataFrame(per_year_rows)
matched_pool = pd.concat(all_matched, ignore_index=True) if all_matched else pd.DataFrame()

print("=== Per-year head-to-head: stats-only XGBoost vs MLB Pipeline (vs actual draft order) ===\n")
display(per_year_h2h_df.round(3))

if len(matched_pool) > 0:
    n_total = len(matched_pool)
    sp_m_avg = per_year_h2h_df['spearman_model'].mean()
    sp_p_avg = per_year_h2h_df['spearman_pipeline'].mean()
    mae_m_avg = per_year_h2h_df['mae_model'].mean()
    mae_p_avg = per_year_h2h_df['mae_pipeline'].mean()
    print(f"\nMean-across-years (n_total_matched = {n_total}):")
    print(f"  Your Model    Spearman = {sp_m_avg:.3f} | MAE = {mae_m_avg:.1f}")
    print(f"  MLB Pipeline  Spearman = {sp_p_avg:.3f} | MAE = {mae_p_avg:.1f}")

In [ ]:
# Developmental-success benchmark (cell: develop-eval).
# Uses post-draft `current_level` as ground truth -- which players actually advanced?
# IMPORTANT: current_level is NEVER used as a training feature; this is evaluation only.
# Restricted to 2021-2023 cohorts (4+ years of development time has elapsed).

LEVEL_ORD = {'ROK': 1, 'A': 2, 'A+': 3, 'AA': 4, 'AAA': 5, 'MLB': 6}

def lvl(x):
    if pd.isna(x):
        return 0
    s = str(x).strip().upper()
    if s in ('', 'NONE', 'NAN'):
        return 0
    return LEVEL_ORD.get(s, 0)

DEV_YEARS = [2021, 2022, 2023]
TOPK = 50
TARGET_LEVEL = LEVEL_ORD['AA']  # "reached AA+"

dev_rows = []
for y in DEV_YEARS:
    yr_oof = oof_results_rank_df[oof_results_rank_df['year'] == y].copy()
    yr_prosp = prospects_by_year[y].copy()
    yr_prosp['level_ord'] = yr_prosp['current_level'].apply(lvl)
    merged = yr_oof.merge(yr_prosp[['name_clean', 'rank', 'level_ord']], on='name_clean', how='inner')

    if len(merged) < TOPK:
        continue

    # Spearman: lower-is-better rankers vs higher-is-better level -> expect NEGATIVE.
    # We negate so "concordance" is positive when the ranker put successful pros at the top.
    sp_model,  _ = spearmanr(merged['Predicted_College_Order'], merged['level_ord'])
    sp_pipe,   _ = spearmanr(merged['rank'],                    merged['level_ord'])
    sp_actual, _ = spearmanr(merged['Actual_College_Order'],    merged['level_ord'])

    def pct_topk(df, ranker_col, k=TOPK, target=TARGET_LEVEL):
        sub = df.sort_values(ranker_col).head(k)
        return 100.0 * (sub['level_ord'] >= target).sum() / len(sub)

    dev_rows.append({
        'year': y,
        'n_matched': len(merged),
        'concord_model_vs_level':    -sp_model,
        'concord_pipeline_vs_level': -sp_pipe,
        'concord_actual_vs_level':   -sp_actual,
        f'pct_AAplus_top{TOPK}_model':    pct_topk(merged, 'Predicted_College_Order'),
        f'pct_AAplus_top{TOPK}_pipeline': pct_topk(merged, 'rank'),
        f'pct_AAplus_top{TOPK}_actual':   pct_topk(merged, 'Actual_College_Order'),
    })

dev_df = pd.DataFrame(dev_rows)
print("=== Developmental-success benchmark (2021-2023 cohorts) ===")
print("Concordance = -Spearman(ranking, level_reached). Higher = ranker put eventual pros at the top.")
print(f"pct_AAplus_top{TOPK} = of the ranker's top {TOPK} players, what % reached AA or higher.\n")
display(dev_df.round(3))

In [ ]:
# V6 summary -- single table for the paper.

print("=" * 78)
print("V6 SUMMARY -- Multi-Year MLB Pipeline Benchmark (stats-only XGBoost)")
print("=" * 78)

print("\n--- Draft-order agreement (Spearman with actual college draft order) ---")
order_summary = per_year_h2h_df[['year', 'n_matched', 'spearman_model', 'spearman_pipeline']].copy()
order_summary.columns = ['year', 'n', 'XGBoost (stats only)', 'MLB Pipeline']
mean_row = pd.DataFrame([{
    'year': 'mean',
    'n': order_summary['n'].sum(),
    'XGBoost (stats only)': order_summary['XGBoost (stats only)'].mean(),
    'MLB Pipeline': order_summary['MLB Pipeline'].mean(),
}])
order_summary = pd.concat([order_summary, mean_row], ignore_index=True)
display(order_summary.round(3))

if len(dev_df) > 0:
    print("\n--- Developmental success (concordance with post-draft level, 2021-2023) ---")
    dev_summary = dev_df[['year', 'n_matched', 'concord_model_vs_level', 'concord_pipeline_vs_level', 'concord_actual_vs_level']].copy()
    dev_summary.columns = ['year', 'n', 'XGBoost', 'MLB Pipeline', 'Actual draft']
    mean_row = pd.DataFrame([{
        'year': 'mean',
        'n': dev_summary['n'].sum(),
        'XGBoost': dev_summary['XGBoost'].mean(),
        'MLB Pipeline': dev_summary['MLB Pipeline'].mean(),
        'Actual draft': dev_summary['Actual draft'].mean(),
    }])
    dev_summary = pd.concat([dev_summary, mean_row], ignore_index=True)
    display(dev_summary.round(3))

    print(f"\n--- % of top-50 that reached AA+ (2021-2023) ---")
    topk_summary = dev_df[['year', 'pct_AAplus_top50_model', 'pct_AAplus_top50_pipeline', 'pct_AAplus_top50_actual']].copy()
    topk_summary.columns = ['year', 'XGBoost', 'MLB Pipeline', 'Actual draft']
    mean_row = pd.DataFrame([{
        'year': 'mean',
        'XGBoost': topk_summary['XGBoost'].mean(),
        'MLB Pipeline': topk_summary['MLB Pipeline'].mean(),
        'Actual draft': topk_summary['Actual draft'].mean(),
    }])
    topk_summary = pd.concat([topk_summary, mean_row], ignore_index=True)
    display(topk_summary.round(1))

---
## Stage 3 - bonus / slot ratio

Earlier versions fitted two dollar models and divided one prediction by the other, which compounds
their errors and leaves no ratio to score against. This predicts the ratio directly and reconstructs
dollars from it.

The denominator is the published slot value for rounds 1-10 and compensation rounds, and the
bonus-pool exemption threshold after that. Those are different economic objects, so an `is_post10`
flag separates them -- but that flag is nearly a binary split of the Stage 2 target, so every metric
is also reported by segment.

Non-signers are excluded, so the target is the expected ratio *given* that the player signs.

In [ ]:
# Build the slot denominator: real slot value where one exists, threshold after round 10
def post10_threshold_millions(year):
    """Round-11+ bonus-pool exemption threshold, in millions.

    Corrects MLBStatsAPIDraftDataAccess/enrich_draft_data.py lines 20-31, which returns 0.150 for
    2022. Empirically the modal round-11+ bonus is $125,000 in both 2021 (123 picks) and 2022
    (126 picks), and only reaches $150,000 in 2023 (141 picks).
    """
    if year < 2012:
        return np.nan
    if year <= 2016:
        return 0.100
    if year <= 2022:
        return 0.125
    return 0.150

draftedClean_df['is_post10'] = (draftedClean_df['Round'] >= 11).astype(int)
draftedClean_df['slot_denom'] = draftedClean_df['api_pickValue'].where(
    draftedClean_df['api_pickValue'].notna(),
    draftedClean_df[year_col].map(post10_threshold_millions))

assert draftedClean_df['slot_denom'].notna().all(), 'a pick has no slot denominator'
assert (draftedClean_df['slot_denom'] > 0).all(), 'a non-positive slot denominator'

print("slot denominator source:")
print(f"  official slot (rounds 1-10 + comp): {int(draftedClean_df['api_pickValue'].notna().sum())}")
print(f"  post-round-10 threshold:            {int(draftedClean_df['api_pickValue'].isna().sum())}")
display(draftedClean_df.groupby([year_col, 'is_post10'])['slot_denom']
        .agg(['size', 'min', 'median', 'max']).round(4))

In [ ]:
# Bonus as a fraction of slot, logged so half-slot and double-slot sit equally far from parity
# The ratio lives on draftedClean_df, not just on the modeling subset, because the simulation's
# grading cell needs it for every drafted player in the simulated year.
draftedClean_df['bonus_slot_ratio'] = (draftedClean_df['api_signingBonus']
                                      / draftedClean_df['slot_denom'])
draftedClean_df['log_ratio'] = np.log(
    draftedClean_df['bonus_slot_ratio'].clip(*RATIO_CLIP))

ratio_df = draftedClean_df[draftedClean_df['api_signingBonus'].notna()].copy()

_lo, _hi = RATIO_CLIP
_n_clip = int(((ratio_df['bonus_slot_ratio'] < _lo) | (ratio_df['bonus_slot_ratio'] > _hi)).sum())

print(f"n = {len(ratio_df)}  (excluded {len(draftedClean_df) - len(ratio_df)} non-signers)")
print(f"winsorized {_n_clip} rows to {RATIO_CLIP}")
print(f"raw ratio skew {ratio_df['bonus_slot_ratio'].skew():.2f}  ->  "
      f"log skew {ratio_df['log_ratio'].skew():.2f}")
print(f"min ratio {ratio_df['bonus_slot_ratio'].min():.5f} (strictly positive, so log is safe)")

display(ratio_df.groupby(year_col)['bonus_slot_ratio'].describe().round(3))

print("\nby segment:")
for _lab, _m in [('rounds 1-10', ratio_df['is_post10'] == 0), ('rounds 11+', ratio_df['is_post10'] == 1)]:
    _s = ratio_df.loc[_m, 'bonus_slot_ratio']
    print(f"  {_lab:<12} n={len(_s):<5} median={_s.median():.3f}  mean={_s.mean():.3f}  "
          f"exactly at slot={int(np.isclose(_s, 1.0).sum())}  over slot={int((_s > 1).sum())}")

In [ ]:
# Shape of the ratio target -- this is what justifies the log transform
plot_target_dist(ratio_df['bonus_slot_ratio'], stage=3, name='target_dist',
                 by=ratio_df['is_post10'].values, by_label='post10', ref=1.0, logx=True,
                 year=ratio_df[year_col].values)

In [ ]:
# Model the denominator too, so a predicted ratio can be turned back into dollars. For a known
# draftee the slot is just a (year, Pick) lookup, so this only earns its keep pre-draft.
slot_features = [f for f in pick_features if f != 'api_pickValue']

slot_train = draftedClean_df[draftedClean_df[year_col] < TEST_YEAR]
slot_test  = draftedClean_df[draftedClean_df[year_col] == TEST_YEAR]

slot_model = xgb.XGBRegressor(**RP_REG, eval_metric='mae')
slot_model.fit(slot_train[slot_features], np.log(slot_train['slot_denom'] * 1e6))

slot_pred_dollars   = np.exp(slot_model.predict(slot_test[slot_features]))
slot_actual_dollars = slot_test['slot_denom'].values * 1e6

print(f"Slot denominator model (train n={len(slot_train)}, test {TEST_YEAR} n={len(slot_test)}):")
print(f"  MAE:      ${mean_absolute_error(slot_actual_dollars, slot_pred_dollars):,.0f}")
print(f"  R2:       {r2_score(slot_actual_dollars, slot_pred_dollars):.3f}")
spearman_ci(slot_actual_dollars, slot_pred_dollars)

In [ ]:
# Fit the ratio model
ratio_features = [f for f in pick_features if f != 'api_pickValue'] + ['is_post10']
# api_pickValue is excluded because it IS the denominator of the target. Pick, Round and
# College_Draft_Order were never in pick_features.
assert 'api_pickValue' not in ratio_features
assert not {'Pick', 'Round', 'College_Draft_Order'} & set(ratio_features)

ratio_train = ratio_df[ratio_df[year_col] < TEST_YEAR]
ratio_test  = ratio_df[ratio_df[year_col] == TEST_YEAR]
print(f"train n={len(ratio_train)} (years < {TEST_YEAR}) | test n={len(ratio_test)} ({TEST_YEAR})")

ratio_model = xgb.XGBRegressor(**RP_REG, eval_metric='mae')
ratio_model.fit(ratio_train[ratio_features], ratio_train['log_ratio'])

pred_ratio   = np.exp(ratio_model.predict(ratio_test[ratio_features]))
actual_ratio = ratio_test['bonus_slot_ratio'].clip(*RATIO_CLIP).values

# Robustness check: pseudo-Huber is less sensitive to the long right tail than squared error.
ratio_model_huber = xgb.XGBRegressor(**RP_REG, objective='reg:pseudohubererror',
                                     eval_metric='mae')
ratio_model_huber.fit(ratio_train[ratio_features], ratio_train['log_ratio'])
pred_ratio_huber = np.exp(ratio_model_huber.predict(ratio_test[ratio_features]))

In [ ]:
# Score the ratio against three baselines a model has to beat to be worth anything
def ratio_metrics(actual, pred, label, band=None):
    band = AT_SLOT_BAND if band is None else band
    a, p = np.asarray(actual, float), np.asarray(pred, float)
    def _dir(x):
        lg = np.log(x)
        return np.where(np.abs(lg) < band, 0, np.sign(lg))
    out = {
        'n': len(a),
        'MAE': mean_absolute_error(a, p),
        'R2': r2_score(a, p) if len(a) > 1 else np.nan,
        'Spearman': spearmanr(a, p).correlation if len(a) > 1 else np.nan,
        'dir_acc': float((_dir(a) == _dir(p)).mean()),
        'over_slot_prec': (precision_score(a > 1, p > 1, zero_division=0)),
        'over_slot_rec': (recall_score(a > 1, p > 1, zero_division=0)),
        'within_10%': float((np.abs(p / a - 1) <= 0.10).mean()),
    }
    return pd.Series(out, name=label)

_rows = [ratio_metrics(actual_ratio, pred_ratio, 'ratio model (squared error)'),
         ratio_metrics(actual_ratio, pred_ratio_huber, 'ratio model (pseudo-Huber)')]

# Baselines. B3 is the honest bar: a model that only learns "round 11+ signs at slot" is doing
# nothing, and because 54% of round-11+ picks sit at exactly 1.0 it would still post a
# respectable aggregate R2.
_b1 = ratio_train.groupby(year_col)['bonus_slot_ratio'].median().reindex(
    ratio_test[year_col]).fillna(ratio_train['bonus_slot_ratio'].median()).values
_seg_med = ratio_train.groupby('is_post10')['bonus_slot_ratio'].median()
_b3 = ratio_test['is_post10'].map(_seg_med).values
_rows += [ratio_metrics(actual_ratio, _b1, 'B1 per-year median'),
          ratio_metrics(actual_ratio, np.ones_like(actual_ratio), 'B2 constant 1.0'),
          ratio_metrics(actual_ratio, _b3, 'B3 per-segment median')]

print(f"Stage 3 - test year {TEST_YEAR}, all picks")
display(pd.DataFrame(_rows).round(3))

print("\nSegmented (immune to the is_post10 leak: the indicator is constant within a segment)")
for _lab, _m in [('rounds 1-10', ratio_test['is_post10'].values == 0),
                 ('rounds 11+',  ratio_test['is_post10'].values == 1)]:
    if _m.sum() == 0:
        continue
    _seg = [ratio_metrics(actual_ratio[_m], pred_ratio[_m], f'{_lab}: model'),
            ratio_metrics(actual_ratio[_m], _b3[_m], f'{_lab}: B3 segment median')]
    display(pd.DataFrame(_seg).round(3))

spearman_ci(actual_ratio, pred_ratio)

In [ ]:
# Turn the predicted ratio back into dollars
# Predict the ratio, then multiply by the estimated denominator
_denom_pred = np.exp(slot_model.predict(ratio_test[slot_features]))
pred_bonus_dollars   = pred_ratio * _denom_pred
actual_bonus_dollars = ratio_test['api_signingBonus'].values * 1e6

print(f"Reconstructed bonus (test year {TEST_YEAR}, n={len(ratio_test)}):")
print(f"  MAE:      ${mean_absolute_error(actual_bonus_dollars, pred_bonus_dollars):,.0f}")
print(f"  R2:       {r2_score(actual_bonus_dollars, pred_bonus_dollars):.3f}")
spearman_ci(actual_bonus_dollars, pred_bonus_dollars)

def _dir_label(x):
    lg = np.log(x)
    return np.where(np.abs(lg) < AT_SLOT_BAND, 'at slot', np.where(lg > 0, 'over', 'under'))

ratio_results_df = pd.DataFrame({
    'name': ratio_test['nameascii'].values,
    'Pick': ratio_test[target_pick].values,
    'Round': ratio_test['Round'].values,
    'Actual_Ratio': actual_ratio.round(3),
    'Pred_Ratio': pred_ratio.round(3),
    'Actual_Dir': _dir_label(actual_ratio),
    'Pred_Dir': _dir_label(pred_ratio),
    'Actual_Bonus': actual_bonus_dollars,
    'Pred_Bonus': pred_bonus_dollars,
    'Slot': ratio_test['slot_denom'].values * 1e6,
}).sort_values('Pick').reset_index(drop=True)
ratio_results_df['Dir_OK'] = ratio_results_df['Actual_Dir'] == ratio_results_df['Pred_Dir']

_disp = ratio_results_df.head(30).copy()
for _c in ['Actual_Bonus', 'Pred_Bonus', 'Slot']:
    _disp[_c] = _disp[_c].apply(lambda v: f"${v:,.0f}")
display(_disp)

print("\nBiggest over-slot signings the model identified (actual ratio, top 15):")
display(ratio_results_df.nlargest(15, 'Actual_Ratio')[
    ['name', 'Pick', 'Actual_Ratio', 'Pred_Ratio', 'Actual_Dir', 'Pred_Dir', 'Dir_OK']])

In [ ]:
# Show top features by importance for both Stage 3 models
_ = gain_table(ratio_model, label='Stage 3 ratio model')
_ = gain_table(slot_model, label='Stage 3 slot denominator model', top=25)

In [ ]:
# Top-10 feature importance for the ratio model
plot_feature_importance(ratio_model, stage=3)

In [ ]:
# SHAP summary for the ratio model
plot_shap_summary(ratio_model, ratio_train[ratio_features], stage=3)

In [ ]:
# Predicted vs actual ratio on the held-out year
plot_pred_vs_actual(actual_ratio, pred_ratio, stage=3, name='pred_vs_actual',
                    xlabel='Actual bonus / slot', ylabel='Predicted bonus / slot',
                    log=True, ref=1.0,
                    title=f'Stage 3 - bonus/slot ratio ({TEST_YEAR})')

In [ ]:
# Residuals split by draft segment, since the two halves have different denominators
plot_residuals(actual_ratio, pred_ratio, stage=3, hue=ratio_test['is_post10'].values,
               hue_label='post10', xlabel='Predicted bonus / slot')

In [ ]:
# How each top feature moves the predicted bonus/slot ratio
plot_pdp_grid(ratio_model, ratio_train[ratio_features], stage=3)

In [ ]:
# Two-feature interactions for the ratio model
plot_pdp_2d(ratio_model, ratio_train[ratio_features], stage=3)

In [ ]:
# Pitching profile coloured by bonus/slot ratio
plot_pcp(ratio_df, ['rpi_team', 'era_pitch', 'whip_pitch', 'k/9_pitch',
                    'so_pitch', 'ip_pitch'],
         'bonus_slot_ratio', stage=3, name='pcp',
         title='Stage 3 - Pitching profile by bonus/slot ratio')

In [ ]:
# Binned predicted vs actual ratio, to check the model is not just predicting the average
# Binned predicted vs actual ratio per segment -- shows whether the model is just
# predicting the average.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (lab, m) in zip(axes, [('rounds 1-10', ratio_test['is_post10'].values == 0),
                               ('rounds 11+',  ratio_test['is_post10'].values == 1)]):
    if m.sum() < 10:
        ax.axis('off'); continue
    _p, _a = pred_ratio[m], actual_ratio[m]
    _q = pd.qcut(pd.Series(_p), min(10, max(2, m.sum() // 8)), duplicates='drop')
    _g = pd.DataFrame({'p': _p, 'a': _a, 'q': _q}).groupby('q', observed=True).mean()
    _lo, _hi = float(min(_g.p.min(), _g.a.min())), float(max(_g.p.max(), _g.a.max()))
    ax.plot([_lo, _hi], [_lo, _hi], 'r--', lw=1.2, label='perfect')
    ax.plot(_g.p, _g.a, 'o-', color=NAVY, lw=2, label='binned mean')
    ax.axhline(1.0, color='gray', ls=':', lw=1); ax.axvline(1.0, color='gray', ls=':', lw=1)
    ax.set_title(f'{lab} (n={int(m.sum())})', fontsize=12)
    ax.set_xlabel('Mean predicted ratio'); ax.set_ylabel('Mean actual ratio')
    ax.grid(ls=':', lw=0.5, alpha=0.6); ax.legend(fontsize=10)
fig.suptitle('Stage 3 - Ratio Calibration by Segment', fontsize=14)
plt.tight_layout()
save_fig(fig, 3, 'ratio_calibration')
plt.show()

### Two-part specification (ablation)

The ratio target is a mixture: a spike at exactly slot plus a continuous body below it. In rounds
1-10, 169 of 1,179 signed picks land at exactly 1.00. One regressor cannot fit a spike and a
continuum under a single loss, so the alternative is two parts - a classifier for "signs at slot", a
regressor for everyone else, and a gate to choose between them.

Scored out-of-fold across every year rather than on 2026 alone, and restricted to rounds 1-10 where
the denominator is a real slot value and no `is_post10` flag is needed.

In [ ]:
# Compare the one-model and two-part specifications out-of-fold across every year
# Scored out-of-fold: for each year, train on the other five and predict it. That puts the
# comparison on 1,179 predictions instead of one test year.

_r1 = ratio_df[ratio_df['is_post10'] == 0].copy().reset_index(drop=True)
_r1['at_slot'] = np.isclose(_r1['bonus_slot_ratio'], 1.0).astype(int)

# is_post10 is constant within this scope, so it carries no information and -- usefully -- the
# leakage caveat attached to it disappears entirely here.
ratio_features_r1 = [f for f in ratio_features if f != 'is_post10']
_pa_feats = [f for f in PARTA_FEATURES if f in ratio_features_r1]
assert len(_pa_feats) >= 10, f'Part A feature list resolved to only {_pa_feats}'
_yrs = sorted(_r1[year_col].unique())
print(f"rounds 1-10: n={len(_r1)}, at slot {int(_r1['at_slot'].sum())} "
      f"({_r1['at_slot'].mean():.1%}) | Part A features: {len(_pa_feats)}")

_oof = {k: np.full(len(_r1), np.nan) for k in ('single', 'partB', 'pA', 'median')}
for _y in _yrs:
    _trm = (_r1[year_col] != _y).values
    _tem = ~_trm
    _tr, _te = _r1[_trm], _r1[_tem]

    # reference: one regressor over everybody (the headline specification)
    _s = xgb.XGBRegressor(**RP_REG, objective='reg:pseudohubererror', eval_metric='mae')
    _s.fit(_tr[ratio_features_r1], _tr['log_ratio'])
    _oof['single'][_tem] = np.exp(_s.predict(_te[ratio_features_r1]))

    # Part A: at-slot classifier
    _ytr = _tr['at_slot'].values
    _a = xgb.XGBClassifier(**PARTA_PARAMS, eval_metric='logloss',
                           scale_pos_weight=(len(_ytr) - _ytr.sum()) / max(_ytr.sum(), 1))
    _a.fit(_tr[_pa_feats], _ytr)
    _oof['pA'][_tem] = _a.predict_proba(_te[_pa_feats])[:, 1]

    # Part B: magnitude, fit only where the player did NOT sign at slot
    _nb = _tr[_tr['at_slot'] == 0]
    _b = xgb.XGBRegressor(**RP_REG, objective='reg:pseudohubererror', eval_metric='mae')
    _b.fit(_nb[ratio_features_r1], _nb['log_ratio'])
    _oof['partB'][_tem] = np.exp(_b.predict(_te[ratio_features_r1]))

    _oof['median'][_tem] = _tr['bonus_slot_ratio'].median()

_act = _r1['bonus_slot_ratio'].clip(*RATIO_CLIP).values
print(f"\nPart A out-of-fold: PR-AUC {average_precision_score(_r1['at_slot'], _oof['pA']):.3f} "
      f"(baseline {_r1['at_slot'].mean():.3f})  "
      f"ROC-AUC {roc_auc_score(_r1['at_slot'], _oof['pA']):.3f}")

_rows = [ratio_metrics(_act, _oof['single'], 'single regressor (pseudo-Huber)')]
for _t in PARTA_GATES:
    _rows.append(ratio_metrics(_act, np.where(_oof['pA'] >= _t, 1.0, _oof['partB']),
                               f'two-part (gate {_t:.2f})'))
_rows += [ratio_metrics(_act, _oof['median'], 'baseline: per-year median'),
          ratio_metrics(_act, np.ones(len(_r1)), 'baseline: constant 1.0')]
print("\nRounds 1-10, out-of-fold over all years:")
display(pd.DataFrame(_rows).round(3))
print("Read the columns as competing objectives, not one score: snapping a prediction to exactly\n"
      "1.0 buys absolute accuracy and loses ranking information.")

In [ ]:
# Per-year error, and the two-part predictions in ratio space
_GATE = 0.30                      # lowest MAE in the sweep above
_comb = np.where(_oof['pA'] >= _GATE, 1.0, _oof['partB'])

_per = []
for _y in _yrs:
    _m = (_r1[year_col] == _y).values
    _per.append({'year': int(_y), 'n': int(_m.sum()),
                 'single': mean_absolute_error(_act[_m], _oof['single'][_m]),
                 'two_part': mean_absolute_error(_act[_m], _comb[_m]),
                 'median': mean_absolute_error(_act[_m], _oof['median'][_m])})
_per = pd.DataFrame(_per).set_index('year')
_per['winner'] = _per[['single', 'two_part', 'median']].idxmin(axis=1)
print(f"Per-year MAE (gate {_GATE:.2f}):")
display(_per.round(3))
print(f"  two-part wins {int((_per['winner']=='two_part').sum())} of {len(_per)} years, "
      f"single {int((_per['winner']=='single').sum())}, median {int((_per['winner']=='median').sum())}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
_w = 0.26
_x = np.arange(len(_per))
for _k, (_c, _col) in enumerate([('single', NAVY), ('two_part', RED), ('median', '#999999')]):
    axes[0].bar(_x + (_k - 1) * _w, _per[_c], _w, label=_c.replace('_', '-'), color=_col, zorder=3)
axes[0].set_xticks(_x); axes[0].set_xticklabels(_per.index)
axes[0].set_ylabel('MAE (ratio)'); axes[0].set_xlabel('held-out year')
axes[0].legend(fontsize=10); axes[0].grid(axis='y', ls=':', lw=0.5, alpha=0.6, zorder=0)
axes[0].set_title('Out-of-fold error by year', fontsize=12)

axes[1].scatter(_act, _comb, alpha=0.5, s=22, color=NAVY, edgecolor='none')
axes[1].plot([0.01, 4], [0.01, 4], 'r--', lw=1.2, label='perfect')
axes[1].axhline(1.0, color='gray', ls=':', lw=1); axes[1].axvline(1.0, color='gray', ls=':', lw=1)
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_xlabel('Actual bonus / slot'); axes[1].set_ylabel('Predicted bonus / slot')
axes[1].set_title(f'Two-part predictions (gate {_GATE:.2f})', fontsize=12)
axes[1].grid(ls=':', lw=0.5, alpha=0.5); axes[1].legend(fontsize=10, loc='upper left')
fig.suptitle('Stage 3 - Two-Part Specification (rounds 1-10, out-of-fold)', fontsize=14)
plt.tight_layout()
save_fig(fig, 3, 'twopart')
plt.show()

plot_pr_curve(_r1['at_slot'].values, _oof['pA'], stage=3, name='partA_pr_curve',
              label='Part A: signs at slot')

In [ ]:
# Dollars, log-log -- kept under its original filename (s3_scatter) since the paper cites it.
_eps = 1.0
plot_pred_vs_actual(np.maximum(actual_bonus_dollars, _eps),
                    np.maximum(pred_bonus_dollars, _eps),
                    stage=3, name='scatter', log=True,
                    xlabel='Actual signing bonus ($)', ylabel='Predicted signing bonus ($)',
                    title=f'Stage 3 - reconstructed bonus ({TEST_YEAR})')

---
## Scouting Report Explainability Layer

For any player or custom stat line, the model now emits a **scouting report**:
a letter grade per dimension (Draft Probability and Draft Order Rank), a short
narrative, and ranked strengths/concerns showing which stats helped or hurt
the prediction.

Two explainers run side-by-side:

1. **SHAP TreeExplainer** — exact per-feature contribution to each prediction
   (gold-standard for XGBoost). Skipped automatically if `shap` is not installed.
2. **Importance × z-score** — XGBoost gain importance × `(player - train_median) / train_std`.
   No extra dependencies; works as a cross-check.

Scope: Draft Probability classifier (`draft_model`) and Draft Order Rank
regressor (`model_rank`). Bonus and slot models are out of scope here.

In [ ]:
# Scouting Report -- explainer setup (run once)
# Builds the SHAP TreeExplainers (if shap is available), normalized feature
# importance dicts, training-set distributions for grading, a human label map,
# and a team-metadata frame re-loaded from the raw CSV (since cell 2 drops the
# team/Full Team Name/league_team columns from df to keep it numeric).

# Probe shap for real rather than assuming the import worked
try:
    _ = shap.TreeExplainer(draft_model)
    _HAS_SHAP = True
except Exception as _e:
    _HAS_SHAP = False
    print(f"shap unavailable ({_e.__class__.__name__}) -- falling back to importance x z-score only.")

# Threshold below which the rank/bonus/slot predictions are suppressed in the
# scouting report -- the rank model is trained only on drafted players, so for
# clearly-not-drafted profiles those numbers are unreliable extrapolation.
MIN_DRAFT_PROB_FOR_PICK_DETAILS = 0.25

# Re-load just the team-metadata columns from the raw CSV. df has these dropped
# (cell 2) so it stays numeric for the models; we need them for the scouting header.
# Re-load the team-metadata columns, which are dropped from df to keep it numeric
_team_meta = pd.read_csv(
    DATA_FILE,
    delimiter=',',
    usecols=['nameascii', 'year', 'team', 'Full Team Name', 'league_team'],
)
_team_meta_idx = (
    _team_meta.dropna(subset=['nameascii', 'year'])
              .drop_duplicates(subset=['nameascii', 'year'], keep='last')
              .set_index(['nameascii', 'year'])
)

def _get_player_team_meta(nameascii, year):
    """Return {team, Full Team Name, league_team} for (nameascii, year), or Nones."""
    try:
        row = _team_meta_idx.loc[(nameascii, int(year))]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        return {
            'team': row.get('team'),
            'Full Team Name': row.get('Full Team Name'),
            'league_team': row.get('league_team'),
        }
    except (KeyError, TypeError):
        return {'team': None, 'Full Team Name': None, 'league_team': None}

# Snapshot the *_team stat columns at setup time -- the Carbon Tracking cell at
# the end of the notebook reassigns df = pd.read_csv(emissions.csv), so any later
# call into get_team_stats would fail with KeyError on df['nameascii']. Build a
# frozen (team_lower, year) -> {*_team col: value} lookup right here, while df
# still points at the player frame.
_team_stat_cols = [c for c in df.columns if c.endswith('_team')]
_TEAM_STATS_BY_TEAM_YEAR = {}
for (_tc, _yr), _grp in (_team_meta
                         .dropna(subset=['nameascii', 'year', 'team'])
                         .groupby(['team', 'year'])):
    _na = _grp['nameascii'].iloc[0]
    _m = df[(df['nameascii'] == _na) & (df[year_col] == _yr)]
    if _m.empty:
        continue
    _row = _m.iloc[0]
    _TEAM_STATS_BY_TEAM_YEAR[(str(_tc).lower(), int(_yr))] = {
        col: _row[col] for col in _team_stat_cols
    }

# Training-set summary stats on the same split used for train_medians.
train_stds = df[df[year_col] < test_year].std(numeric_only=True)

# TreeExplainers (operate on raw model objects; very fast for XGBoost).
expl_draft = shap.TreeExplainer(draft_model) if _HAS_SHAP else None
expl_rank  = shap.TreeExplainer(model_rank)  if _HAS_SHAP else None

# Normalized gain-importance dicts (sum to 1.0).
def _normed_importance(model):
    raw = model.get_booster().get_score(importance_type='gain')
    total = sum(raw.values()) or 1.0
    return {k: v / total for k, v in raw.items()}

imp_draft = _normed_importance(draft_model)
imp_rank  = _normed_importance(model_rank)

# Training distributions for letter-grading the predictions.
_train_mask     = df[year_col] < test_year
_train_X_cls    = df.loc[_train_mask, features]
_train_drafted  = draftedClean_df[draftedClean_df[year_col] < test_year]
_train_X_rank   = _train_drafted[pick_features]
_train_draft_probs = draft_model.predict_proba(_train_X_cls)[:, 1]
_train_rank_preds  = model_rank.predict(_train_X_rank)

# Human-readable labels (raw column name -> short label).
FEATURE_LABELS = {
    'age': 'Age', 'role': 'Role',
    # Pitching
    'era_pitch': 'ERA',         'fip_pitch': 'FIP',
    'k/9_pitch': 'K/9',         'bb/9_pitch': 'BB/9',
    'k/bb_pitch': 'K/BB',       'whip_pitch': 'WHIP',
    'k%_pitch': 'K%',           'bb%_pitch': 'BB%',
    'k-bb%_pitch': 'K-BB%',     'ip_pitch': 'IP',
    'so_pitch': 'SO',           'bb_pitch': 'BB (pitch)',
    'h_pitch': 'H allowed',     'hr_pitch': 'HR allowed',
    'hr/9_pitch': 'HR/9',       'er_pitch': 'ER',
    'r_pitch': 'R allowed',     'avg_pitch': 'BA against',
    'babip_pitch': 'BABIP-against', 'lob%_pitch': 'LOB%',
    'e-f_pitch': 'ERA-FIP',     'sv_pitch': 'SV',
    'g_pitch': 'G (pitch)',     'gs_pitch': 'GS',
    'w_pitch': 'W',             'l_pitch': 'L',
    'tbf_pitch': 'TBF',         'hbp_pitch': 'HBP (pitch)',
    'wp_pitch': 'WP',           'bk_pitch': 'BK',
    'cg_pitch': 'CG',           'sho_pitch': 'SHO',
    # Batting
    'avg_bat': 'AVG',  'obp_bat': 'OBP',  'slg_bat': 'SLG',  'ops_bat': 'OPS',
    'iso_bat': 'ISO',  'woba_bat': 'wOBA','wrc+_bat': 'wRC+','wrc_bat': 'wRC',
    'wraa_bat': 'wRAA','hr_bat': 'HR',    'rbi_bat': 'RBI',  'r_bat': 'R',
    'h_bat': 'H',      '2b_bat': '2B',    '3b_bat': '3B',    '1b_bat': '1B',
    'bb%_bat': 'BB% (bat)',  'k%_bat': 'K% (bat)',
    'bb_bat': 'BB (bat)',    'so_bat': 'SO (bat)',
    'sb_bat': 'SB',          'cs_bat': 'CS',
    'babip_bat': 'BABIP',    'spd_bat': 'Spd',
    'g_bat': 'G (bat)',      'ab_bat': 'AB',  'pa_bat': 'PA',
    'hbp_bat': 'HBP (bat)',  'sf_bat': 'SF',  'sh_bat': 'SH',  'gdp_bat': 'GDP',
    'bb/k_bat': 'BB/K',      'wsb_bat': 'wSB',
}


def get_team_stats(team, team_year):
    """Return all *_team stats for a (team, team_year) pair, plus resolved metadata.

    Strict: `team` matches the raw-CSV `team` column case-insensitively (e.g. 'LSU',
    'USF'); `team_year` must exactly match `year`. Reads from the setup-time
    snapshot `_TEAM_STATS_BY_TEAM_YEAR` (not from df), so it stays correct even
    after the Carbon Tracking cell reassigns df to the emissions frame.

    Returns a flat dict for predict_from_stats(..., team_stats_dict=...). The
    `_resolved_*` keys carry display metadata and are ignored by the predictor.

    Raises ValueError if no match is found.
    """
    if team is None or team_year is None:
        raise ValueError("get_team_stats requires both `team` and `team_year`.")
    key = (str(team).lower(), int(team_year))
    stats = _TEAM_STATS_BY_TEAM_YEAR.get(key)
    if stats is None:
        codes = sorted({k[0].upper() for k in _TEAM_STATS_BY_TEAM_YEAR})[:25]
        raise ValueError(
            f"No rows found for team={team!r} year={team_year}. "
            f"Try one of (first 25 codes): {codes}. "
            f"Full list: sorted({{k[0].upper() for k in _TEAM_STATS_BY_TEAM_YEAR}})."
        )

    mask = (_team_meta['team'].astype(str).str.lower() == str(team).lower()) & (_team_meta['year'] == team_year)
    meta = _team_meta[mask].iloc[0]

    out = dict(stats)
    out['_resolved_team'] = str(meta['team']).upper()
    out['_resolved_team_year'] = int(team_year)
    full_name = meta.get('Full Team Name')
    out['_resolved_full_name'] = str(full_name).title() if pd.notna(full_name) else out['_resolved_team']
    league = meta.get('league_team')
    out['_resolved_conference'] = str(league) if pd.notna(league) else None
    return out


print("Explainer setup loaded.")
print(f"  SHAP available:        {_HAS_SHAP}")
print(f"  draft features:        {len(features)}    rank features: {len(pick_features)}")
print(f"  train baseline draft%: {_train_draft_probs.mean()*100:.1f}%")
print(f"  train rank dist:       median={np.median(_train_rank_preds):.1f}  "
      f"p10={np.percentile(_train_rank_preds,10):.1f}  p90={np.percentile(_train_rank_preds,90):.1f}")
print(f"  pick-detail threshold: {MIN_DRAFT_PROB_FOR_PICK_DETAILS*100:.0f}% draft prob (below this, rank/bonus/slot are hidden)")
print(f"  team-meta rows loaded: {len(_team_meta):,}  (re-loaded from CSV; df has these columns dropped)")
print(f"  team-stats snapshot:   {len(_TEAM_STATS_BY_TEAM_YEAR):,} (team, year) pairs cached (survives df mutations)")
print(f"  get_team_stats() ready -- usage: get_team_stats('LSU', 2024)")

In [ ]:
# Scouting Report -- core functions
#   _explain_one(row, kind)  -> per-feature contribution records
#   _grade(percentile)       -> letter grade
#   scouting_report(row, draft_prob, rank_pred, name=..., team=..., conference=...)

def _label(feat):
    return FEATURE_LABELS.get(feat, feat)

def _fmt(value):
    if value is None: return '-'
    if isinstance(value, float):
        if np.isnan(value) or np.isinf(value): return '-'
        return f"{value:.3f}" if abs(value) < 10 else f"{value:.1f}"
    return str(value)

def _rec_score(r):
    """Single sortable score per record: prefer SHAP, fall back to imp_z."""
    for k in ('shap_pct', 'shap_picks'):
        if r[k] is not None and not (isinstance(r[k], float) and np.isnan(r[k])):
            return r[k]
    return r['imp_z']

def _explain_one(row_series, model_kind):
    """Return contribution records sorted strengths-first (positive = good for the player)."""
    if model_kind == 'draft':
        feat_list, imp, explainer, sign_flip = features, imp_draft, expl_draft, 1.0
    elif model_kind == 'rank':
        feat_list, imp, explainer, sign_flip = pick_features, imp_rank, expl_rank, -1.0
    else:
        raise ValueError(model_kind)

    vals = row_series.reindex(feat_list).values
    x_df = pd.DataFrame([vals], columns=feat_list)

    shap_vals = None
    base = 0.0
    log_pred = 0.0
    if explainer is not None:
        sv = explainer.shap_values(x_df)
        if isinstance(sv, list):
            sv = sv[1]
        shap_vals = sv[0]
        ev = explainer.expected_value
        try:
            base = float(ev) if not hasattr(ev, '__len__') else float(ev[1] if len(ev) > 1 else ev[0])
        except (TypeError, ValueError):
            base = float(np.asarray(ev).ravel()[-1])
        log_pred = base + float(np.nansum(shap_vals))

    recs = []
    for i, feat in enumerate(feat_list):
        v = row_series.get(feat)
        if v is None or (isinstance(v, float) and pd.isna(v)):
            continue
        med = train_medians.get(feat, np.nan)
        std = train_stds.get(feat, np.nan)
        imp_z = 0.0 if (pd.isna(std) or std == 0) else imp.get(feat, 0.0) * float((v - med) / std) * sign_flip

        shap_pct = shap_picks = None
        if shap_vals is not None:
            s = float(shap_vals[i])
            if model_kind == 'draft':
                shap_pct = (expit(log_pred) - expit(log_pred - s)) * 100.0
            else:
                shap_picks = s * sign_flip

        recs.append({
            'feature': feat, 'label': _label(feat),
            'value': v, 'median': med,
            'shap_pct': shap_pct, 'shap_picks': shap_picks,
            'imp_z': imp_z,
        })

    recs.sort(key=_rec_score, reverse=True)
    return recs


def _draft_grade(draft_prob):
    """Map raw draft probability directly to a letter grade.

    Absolute thresholds (not training percentiles), so 0% -> F instead of D+:
        >= 90% A+   >= 80% A    >= 70% A-
        >= 60% B+   >= 50% B    >= 40% B-
        >= 30% C+   >= 20% C    >= 10% C-
        >=  5% D+   >=  2% D    >=  1% D-
        <   1% F
    Returns (letter, training_pct) -- training_pct is still computed so the report
    can show it as an auxiliary stat in parentheses.
    """
    p_pct = draft_prob * 100
    bands = [(90,'A+'),(80,'A'),(70,'A-'),(60,'B+'),(50,'B'),(40,'B-'),
             (30,'C+'),(20,'C'),(10,'C-'),(5,'D+'),(2,'D'),(1,'D-')]
    letter = 'F'
    for thresh, lbl in bands:
        if p_pct >= thresh:
            letter = lbl
            break
    train_pct = float((_train_draft_probs <= draft_prob).mean() * 100)
    return letter, train_pct


def _rank_grade(rank_pred):
    pct = float((_train_rank_preds >= rank_pred).mean() * 100)
    bins = [(95,'A+'),(88,'A'),(80,'A-'),(72,'B+'),(62,'B'),(54,'B-'),
            (46,'C+'),(38,'C'),(30,'C-'),(22,'D+'),(15,'D'),(8,'D-')]
    letter = 'F'
    for t, lbl in bins:
        if pct >= t:
            letter = lbl
            break
    return letter, pct


def _narrative(draft_letter, top_strength, top_concern, rank_pred, show_rank):
    band = {
        'A+':'a top profile', 'A':'an elite projection',
        'A-':'a strong projection', 'B+':'a sturdy projection',
        'B':'a solid projection', 'B-':'a mid projection',
        'C+':'a back-half projection', 'C':'a fringe projection',
        'C-':'a fringe-draftable profile', 'D+':'a long-shot draft profile',
        'D':'an undrafted-leaning profile', 'D-':'a Undrafted Free Agent (UDFA)-leaning profile',
        'F':'a clearly undrafted-leaning profile',
    }.get(draft_letter, 'a draftable profile')
    rank_phrase = f" (~pick {int(round(rank_pred))})" if show_rank else ""
    parts = [f"The model sees {band}{rank_phrase}."]
    if top_strength is not None:
        parts.append(f"Driven by {top_strength['label']} ({_fmt(top_strength['value'])} vs {_fmt(top_strength['median'])} median).")
    if top_concern is not None:
        parts.append(f"Main red flag: {top_concern['label']} ({_fmt(top_concern['value'])} vs {_fmt(top_concern['median'])} median).")
    return " ".join(parts)


def scouting_report(row_series, draft_prob, rank_pred, name=None, team=None, conference=None, top_n=5):
    """Print the scouting report block.

    row_series: pandas Series indexed by feature name (player or custom row).
    team/conference: optional strings shown right under the player name.
    """
    name = name or 'Custom Stat Line'
    show_rank = draft_prob >= MIN_DRAFT_PROB_FOR_PICK_DETAILS

    d_letter, d_pct = _draft_grade(draft_prob)
    if show_rank:
        r_letter, r_pct = _rank_grade(rank_pred)

    draft_recs = _explain_one(row_series, 'draft')
    rank_recs  = _explain_one(row_series, 'rank') if show_rank else []

    strengths = [r for r in draft_recs if _rec_score(r) > 0][:top_n]
    concerns  = [r for r in reversed(draft_recs) if _rec_score(r) < 0][:top_n]
    top_s = strengths[0] if strengths else None
    top_c = concerns[0] if concerns else None

    sep = '=' * 64
    print(f"\n{sep}")
    print(f"  Scouting Report: {name}")
    if team or conference:
        parts = []
        if team:       parts.append(f"Team: {team}")
        if conference: parts.append(f"Conference: {conference}")
        print(f"  {'   '.join(parts)}")
    print(sep)
    print(f"  Draft Grade: {d_letter}   (P[draft]={draft_prob*100:.1f}%, "
          f"{d_pct:.0f}th pct of training preds)")
    if show_rank:
        print(f"  Rank  Grade: {r_letter}   (predicted pick ~{int(round(rank_pred))}, "
              f"{r_pct:.0f}th pct of training preds)")
    else:
        print(f"  Rank  Grade: -    (suppressed: draft prob < {MIN_DRAFT_PROB_FOR_PICK_DETAILS*100:.0f}%)")
    print('-' * 64)
    print(f"  Take: {_narrative(d_letter, top_s, top_c, rank_pred, show_rank)}")

    def _print_block(title, recs, arrow):
        print()
        header = "    feature             value      vs median"
        if _HAS_SHAP: header += "      d-prob"
        header += "    imp x z"
        print(f"  {title}")
        print(header)
        if not recs:
            print("    (none)")
            return
        for r in recs:
            shap_part = ''
            if _HAS_SHAP and r['shap_pct'] is not None:
                shap_part = f"   {r['shap_pct']:+6.2f}%"
            elif _HAS_SHAP:
                shap_part = '          -'
            imp_part = f"   {r['imp_z']:+7.3f}"
            print(f"    {arrow} {r['label']:<16s}  {_fmt(r['value']):>8s}  "
                  f"(vs {_fmt(r['median']):>7s}){shap_part}{imp_part}")

    _print_block(f"Top {len(strengths)} STRENGTHS (boost draft stock)", strengths, '^')
    _print_block(f"Top {len(concerns)} CONCERNS  (drag it down)",        concerns,  'v')

    if show_rank:
        r_str = [r for r in rank_recs if _rec_score(r) > 0][:3]
        r_con = [r for r in reversed(rank_recs) if _rec_score(r) < 0][:3]
        print()
        print("  Rank-model cross-check (which features improve / hurt predicted pick):")
        print(f"    + improves pick: {', '.join(r['label'] for r in r_str) or '(none)'}")
        print(f"    - hurts pick:    {', '.join(r['label'] for r in r_con) or '(none)'}")
    print(sep)

---
## Player Lookup by Name

Fuzzy-match a player and show stats, draft probability, predicted college order, and predicted signing bonus.

In [ ]:
# Grading distributions come from the same filtered population the models were fit on
train_medians = df[df[year_col] < test_year].median(numeric_only=True)

def _format_team_display(team_code, full_team_name):
    """Title-case the full team name; append the short code in parens if both exist."""
    has_full = pd.notna(full_team_name) and str(full_team_name).strip()
    has_code = pd.notna(team_code) and str(team_code).strip()
    if has_full and has_code:
        return f"{str(full_team_name).title()} ({str(team_code).upper()})"
    if has_full:
        return str(full_team_name).title()
    if has_code:
        return str(team_code).upper()
    return None

def lookup_player(name_query, year=None, explain=True):
    candidates = df if year is None else df[df[year_col] == year]
    names = candidates['nameascii'].dropna().unique().tolist()
    matches = get_close_matches(name_query, names, n=5, cutoff=0.4)
    if not matches:
        print(f"No match found for '{name_query}'"); return None

    player = candidates[candidates['nameascii'] == matches[0]].sort_values(year_col, ascending=False).iloc[0]
    role_name = {0: 'Pitcher', 1: 'Batter', 2: 'Both'}.get(player['role'], '?')

    # team/Full Team Name/league_team were dropped from df in cell 2 to keep
    # df numeric; re-load them from _team_meta (built in cell 56 from the CSV).
    team_meta = _get_player_team_meta(player['nameascii'], int(player[year_col]))
    team_display = _format_team_display(team_meta['team'], team_meta['Full Team Name'])
    league = team_meta['league_team']
    conf_display = str(league) if pd.notna(league) else None

    print(f"{'='*60}")
    print(f"Player: {player['nameascii']}  |  Year: {int(player[year_col])}  |  Role: {role_name}")
    if team_display or conf_display:
        bits = []
        if team_display: bits.append(f"Team: {team_display}")
        if conf_display: bits.append(f"Conference: {conf_display}")
        print('  ' + '   '.join(bits))
    print(f"{'='*60}")

    if not pd.isna(player.get('api_height')):
        ft, inch = int(player['api_height'] // 12), int(player['api_height'] % 12)
        print(f"Height: {ft}'{inch}\"  |  Weight: {int(player.get('api_weight', 0))} lbs")
    pos_rev = {1:'C', 2:'SS', 3:'2B', 4:'3B', 5:'CF', 6:'LF', 7:'RF', 8:'IF', 9:'1B', 10:'OF', 11:'DH', 12:'P', 13:'TWP'}
    if not pd.isna(player.get('api_primaryPos')):
        print(f"Position: {pos_rev.get(int(player['api_primaryPos']), '?')}")

    print(f"\n--- Key Stats ---")
    if player['role'] in [0, 2]:
        for s in ['era_pitch', 'fip_pitch', 'k/9_pitch', 'bb/9_pitch', 'whip_pitch', 'ip_pitch', 'so_pitch']:
            if not pd.isna(player.get(s)): print(f"  {s}: {player[s]:.2f}")
    if player['role'] in [1, 2]:
        for s in ['avg_bat', 'ops_bat', 'hr_bat', 'wrc+_bat', 'woba_bat', 'slg_bat', 'bb%_bat', 'k%_bat']:
            if not pd.isna(player.get(s)):
                v = player[s]; print(f"  {s}: {v:.3f}" if isinstance(v, float) and v < 10 else f"  {s}: {v:.1f}")

    cls_df = pd.DataFrame([player.reindex(features).values], columns=features)
    draft_prob = draft_model.predict_proba(cls_df)[0][1]
    rank_df = pd.DataFrame([player.reindex(pick_features).values], columns=pick_features)
    rank_pred = model_rank.predict(rank_df)[0]
    # Predict the ratio and reconstruct dollars. The round is unknown for an arbitrary player,
    # so the segment flag comes from the predicted draft order.
    _rr = player.reindex(ratio_features)
    _rr['is_post10'] = 1 if rank_pred > POST10_ORDER_CUT else 0
    ratio = float(np.exp(ratio_model.predict(
        pd.DataFrame([_rr.values], columns=ratio_features).astype(float))[0]))

    slot_df_in = pd.DataFrame([player.reindex(slot_features).values],
                              columns=slot_features).astype(float)
    slot_pred = float(np.exp(slot_model.predict(slot_df_in)[0]))
    bonus_pred = ratio * slot_pred

    pct_pred = model_rank_pct.predict(rank_df)[0]

    print(f"\n--- Predictions ---")
    print(f"Draft Probability: {draft_prob*100:.1f}%")
    if draft_prob >= MIN_DRAFT_PROB_FOR_PICK_DETAILS:
        print(f"Predicted College Draft Order: ~{int(round(rank_pred))}")
        print(f"Predicted College Draft Pct: {pct_pred*100:.1f}%")
        print(f"Predicted Signing Bonus: ${bonus_pred:,.0f}")
        print(f"Predicted Slot Value: ${slot_pred:,.0f}")
        print(f"Bonus/Slot Ratio: {ratio:.2f}")
    else:
        print(f"Predicted draft order / signing bonus / slot value suppressed "
              f"(draft prob {draft_prob*100:.1f}% < {MIN_DRAFT_PROB_FOR_PICK_DETAILS*100:.0f}% threshold -- "
              f"the rank/bonus/slot models are trained only on drafted players and would be extrapolating).")

    if not pd.isna(player.get('Pick')):
        print(f"\n--- Actual ---")
        print(f"Pick: {int(player['Pick'])} (Round {int(player.get('Round', 0))})")
        if not pd.isna(player.get('College_Draft_Order')): print(f"College Order: {int(player['College_Draft_Order'])}")
        if not pd.isna(player.get('api_signingBonus')): print(f"Signing Bonus: ${player['api_signingBonus']*1e6:,.0f}")
    if len(matches) > 1: print(f"\nOther matches: {matches[1:]}")

    if explain and 'scouting_report' in globals():
        scouting_report(player, draft_prob, rank_pred,
                        name=player['nameascii'],
                        team=team_display,
                        conference=conf_display)

    return player

print("lookup_player() defined. Usage: lookup_player('Jace LaViolette')")

In [ ]:
lookup_player('Jace LaViolette')

In [ ]:
lookup_player('Kade Anderson')

In [ ]:
lookup_player('alex mccoy')

In [ ]:
lookup_player('cooper clark')

---
## Custom Stats Input

Predict draft probability, college order, and signing bonus from custom stat values.

In [ ]:
PITCHER_TEMPLATE = {
    'age': 21, 'era_pitch': 3.50, 'fip_pitch': 3.40, 'ip_pitch': 90.0,
    'so_pitch': 100, 'bb_pitch': 25, 'h_pitch': 75, 'hr_pitch': 5,
    'g_pitch': 15, 'gs_pitch': 14, 'w_pitch': 8, 'l_pitch': 3,
    'k/9_pitch': 10.0, 'bb/9_pitch': 2.5, 'k/bb_pitch': 4.0, 'whip_pitch': 1.10,
    'k%_pitch': 0.28, 'bb%_pitch': 0.07, 'k-bb%_pitch': 0.21,
    'avg_pitch': 0.230, 'babip_pitch': 0.290, 'lob%_pitch': 0.72,
    'hr/9_pitch': 0.5, 'e-f_pitch': 0.5, 'sv_pitch': 0, 'tbf_pitch': 350,
    'r_pitch': 40, 'er_pitch': 35, 'hbp_pitch': 5, 'wp_pitch': 3, 'bk_pitch': 0, 'cg_pitch': 0, 'sho_pitch': 0,
}
BATTER_TEMPLATE = {
    'age': 21, 'avg_bat': 0.300, 'ops_bat': 0.900, 'hr_bat': 15,
    'wrc+_bat': 130, 'woba_bat': 0.400, 'slg_bat': 0.520, 'obp_bat': 0.400,
    'bb%_bat': 0.12, 'k%_bat': 0.15, 'iso_bat': 0.200,
    'g_bat': 55, 'ab_bat': 200, 'pa_bat': 240, 'h_bat': 60,
    '1b_bat': 35, '2b_bat': 15, '3b_bat': 2, 'r_bat': 40,
    'rbi_bat': 45, 'bb_bat': 30, 'so_bat': 35, 'hbp_bat': 5,
    'sf_bat': 3, 'sh_bat': 1, 'gdp_bat': 3, 'sb_bat': 10, 'cs_bat': 2,
    'spd_bat': 5.0, 'babip_bat': 0.330, 'wsb_bat': 1.0, 'wrc_bat': 40.0, 'wraa_bat': 15.0, 'bb/k_bat': 0.8,
}

def predict_from_stats(role, age, stats_dict, team_stats_dict=None,
                       team=None, team_year=None, explain=True, name=None):
    """Predict draft outcomes from a custom stat line.

    New optional kwargs:
      team, team_year -- when both are given AND team_stats_dict is None,
        get_team_stats(team, team_year) is called to auto-populate the team
        stats (and to label the scouting header). Strict match.
    """
    role_num = {'Pitcher': 0, 'Batter': 1, 'Both': 2}[role]

    resolved_team_display = None
    resolved_conference = None
    if team_stats_dict is None and team is not None and team_year is not None:
        team_stats_dict = get_team_stats(team, team_year)
        resolved_team_display = (
            f"{team_stats_dict['_resolved_full_name']} ({team_stats_dict['_resolved_team']})"
        )
        resolved_conference = team_stats_dict.get('_resolved_conference')
        print(f"Auto-loaded team stats: {team_stats_dict['_resolved_team']} "
              f"{team_stats_dict['_resolved_team_year']} "
              f"({resolved_conference or 'unknown conf'})")
    elif team is not None or team_year is not None:
        if team is None or team_year is None:
            raise ValueError("Pass both `team` and `team_year` together, or neither.")

    row = pd.Series(index=pick_features, dtype=float)
    row[:] = np.nan
    row['age'] = age; row['role'] = role_num
    for k, v in stats_dict.items():
        if k in row.index: row[k] = v
    if team_stats_dict:
        for k, v in team_stats_dict.items():
            if k in row.index: row[k] = v
    else:
        for f in row.index:
            if pd.isna(row[f]) and f.endswith('_team') and f in train_medians.index:
                row[f] = train_medians[f]

    cls_df = pd.DataFrame([row.reindex(features).values], columns=features)
    draft_prob = draft_model.predict_proba(cls_df)[0][1]
    rank_df = pd.DataFrame([row.reindex(pick_features).values], columns=pick_features)
    rank_pred = model_rank.predict(rank_df)[0]
    # Predict the ratio and reconstruct dollars. The round is unknown for an arbitrary player,
    # so the segment flag comes from the predicted draft order.
    _rr = row.reindex(ratio_features)
    _rr['is_post10'] = 1 if rank_pred > POST10_ORDER_CUT else 0
    ratio = float(np.exp(ratio_model.predict(
        pd.DataFrame([_rr.values], columns=ratio_features).astype(float))[0]))

    slot_df_in = pd.DataFrame([row.reindex(slot_features).values],
                              columns=slot_features).astype(float)
    slot_pred = float(np.exp(slot_model.predict(slot_df_in)[0]))
    bonus_pred = ratio * slot_pred

    pct_pred = model_rank_pct.predict(rank_df)[0]

    print(f"{'='*50}")
    header = f"Custom {role} Prediction (age {age})"
    if resolved_team_display:
        header += f"  --  {resolved_team_display}"
    print(header)
    print(f"{'='*50}")
    print(f"Draft Probability: {draft_prob*100:.1f}%")
    if draft_prob >= MIN_DRAFT_PROB_FOR_PICK_DETAILS:
        print(f"Predicted College Draft Order: ~{int(round(rank_pred))}")
        print(f"Predicted College Draft Pct: {pct_pred*100:.1f}%")
        print(f"Predicted Signing Bonus: ${bonus_pred:,.0f}")
        print(f"Predicted Slot Value: ${slot_pred:,.0f}")
        print(f"Bonus/Slot Ratio: {ratio:.2f}")
    else:
        print(f"Predicted draft order / signing bonus / slot value suppressed "
              f"(draft prob {draft_prob*100:.1f}% < {MIN_DRAFT_PROB_FOR_PICK_DETAILS*100:.0f}% threshold -- "
              f"the rank/bonus/slot models are trained only on drafted players and would be extrapolating).")

    if explain and 'scouting_report' in globals():
        full_row = row.copy()
        for f in features:
            if f not in full_row.index:
                full_row[f] = np.nan
        for k, v in stats_dict.items():
            if k in features: full_row[k] = v
        if team_stats_dict:
            for k, v in team_stats_dict.items():
                if k in features: full_row[k] = v
        full_row['age'] = age; full_row['role'] = role_num
        scouting_report(full_row, draft_prob, rank_pred,
                        name=name or f"Custom {role} (age {age})",
                        team=resolved_team_display,
                        conference=resolved_conference)

    return {'draft_prob': draft_prob, 'pred_order': rank_pred, 'pred_pct': pct_pred,
            'pred_bonus': bonus_pred, 'pred_slot': slot_pred, 'bonus_slot_ratio': ratio}

print("predict_from_stats() defined. Templates: PITCHER_TEMPLATE, BATTER_TEMPLATE")
print("  New: pass team='LSU', team_year=2024 to auto-populate team stats.")

In [ ]:
predict_from_stats('Pitcher', 21, {
    'era_pitch': 2.50, 'fip_pitch': 2.80, 'so_pitch': 120, 'ip_pitch': 95.0,
    'bb_pitch': 20, 'k/9_pitch': 11.4, 'bb/9_pitch': 1.9, 'whip_pitch': 0.95,
    'k%_pitch': 0.32, 'bb%_pitch': 0.06, 'k-bb%_pitch': 0.26,
    'g_pitch': 16, 'gs_pitch': 16, 'w_pitch': 10, 'l_pitch': 2,
})

In [ ]:
predict_from_stats('Batter', 21, {
    'avg_bat': 0.330, 'ops_bat': 1.050, 'hr_bat': 22, 'wrc+_bat': 170,
    'woba_bat': 0.440, 'slg_bat': 0.620, 'obp_bat': 0.430,
    'bb%_bat': 0.14, 'k%_bat': 0.12, 'iso_bat': 0.280,
    'g_bat': 58, 'ab_bat': 210, 'pa_bat': 255, 'h_bat': 69,
    'r_bat': 55, 'rbi_bat': 60, 'sb_bat': 15,
})

In [ ]:
# Cooper Clark
predict_from_stats('Pitcher', 19, {
    'era_pitch': 7.714286, 'fip_pitch': 5.554999, 'so_pitch': 47, 'ip_pitch': 51.1,
    'bb_pitch': 15, 'k/9_pitch': 8.24, 'bb/9_pitch': 2.63, 'whip_pitch': 1.58,
    'k%_pitch': 0.194, 'bb%_pitch': 0.062, 'k-bb%_pitch': 0.132,
    'g_pitch': 13, 'gs_pitch': 11, 'w_pitch': 4, 'l_pitch': 1,
})

In [ ]:
# Test Both
predict_from_stats('Both', 21, {
    'era_pitch': 6.50, 'fip_pitch': 1.503, 'so_pitch': 70, 'ip_pitch': 63.2,
    'bb_pitch': 50, 'h_pitch': 62, 'hr_pitch': 4, 'r_pitch': 59, 'er_pitch': 46,
    'hbp_pitch': 11, 'wp_pitch': 15, 'bk_pitch': 7, 'cg_pitch': 2, 'sho_pitch': 0,
    'g_pitch': 13, 'gs_pitch': 12, 'w_pitch': 6, 'l_pitch': 2,
    'age': 21, 'avg_bat': 0.310, 'ops_bat': 0.982, 'hr_bat': 2, 'slg_bat': 0.500, 'obp_bat': 0.482,
    'g_bat': 34, 'ab_bat': 42, 'h_bat': 13, '2b_bat': 2, '3b_bat': 0, 'r_bat': 12,
    'rbi_bat': 10, 'bb_bat': 14, 'so_bat': 13, 'hbp_bat': 0,
    'sf_bat': 0, 'sh_bat': 0, 'gdp_bat': 0, 'sb_bat': 0,
})

# Only Pitching stats
predict_from_stats('Pitcher', 21, {
    'era_pitch': 6.50, 'fip_pitch': 1.503, 'so_pitch': 70, 'ip_pitch': 63.2,
    'bb_pitch': 50, 'h_pitch': 62, 'hr_pitch': 4, 'r_pitch': 59, 'er_pitch': 46,
    'hbp_pitch': 11, 'wp_pitch': 15, 'bk_pitch': 7, 'cg_pitch': 2, 'sho_pitch': 0,
    'g_pitch': 13, 'gs_pitch': 12, 'w_pitch': 6, 'l_pitch': 2,
    'age': 21,
})

# Only Batting stats
predict_from_stats('Batter', 21, {
    'age': 21, 'avg_bat': 0.310, 'ops_bat': 0.982, 'hr_bat': 2, 'slg_bat': 0.500, 'obp_bat': 0.482,
    'g_bat': 34, 'ab_bat': 42, 'h_bat': 13, '2b_bat': 2, '3b_bat': 0, 'r_bat': 12,
    'rbi_bat': 10, 'bb_bat': 14, 'so_bat': 13, 'hbp_bat': 0,
    'sf_bat': 0, 'sh_bat': 0, 'gdp_bat': 0, 'sb_bat': 0,
})

In [ ]:
# Test Both with team stats USF
predict_from_stats('Both', 21, {
    'era_pitch': 6.50, 'fip_pitch': 1.503, 'so_pitch': 70, 'ip_pitch': 63.2,
    'bb_pitch': 50, 'h_pitch': 62, 'hr_pitch': 4, 'r_pitch': 59, 'er_pitch': 46,
    'hbp_pitch': 11, 'wp_pitch': 15, 'bk_pitch': 7, 'cg_pitch': 2, 'sho_pitch': 0,
    'g_pitch': 13, 'gs_pitch': 12, 'w_pitch': 6, 'l_pitch': 2,
    'age': 21, 'avg_bat': 0.310, 'ops_bat': 0.982, 'hr_bat': 2, 'slg_bat': 0.500, 'obp_bat': 0.482,
    'g_bat': 34, 'ab_bat': 42, 'h_bat': 13, '2b_bat': 2, '3b_bat': 0, 'r_bat': 12,
    'rbi_bat': 10, 'bb_bat': 14, 'so_bat': 13, 'hbp_bat': 0,
    'sf_bat': 0, 'sh_bat': 0, 'gdp_bat': 0, 'sb_bat': 0,
}, team_stats_dict={
    'conf_rpi_team': 0.5173, 'conf_rank_team': 7, 'W_team': 28, 'L_team': 29, 'T_team': 0, 'G_team': 57, 'WPCT_team': 0.491, 'PE_team': 0.474, 'Difference_team': 0.017, 'BB (Batting)_team': 269, 'AB_team': 1857, 'H_team': 488, 'BA_team': 0.263, 'DP_team': 42, 'DPPG_team': 0.74, '2B_team': 102, '2BPG_team': 1.79, 'IP_team': 494.2, 'R (Pitching)_team': 362, 'ER_team': 320, 'ERA_team': 5.82, 'PO_team': 1484, 'A_team': 520, 'E_team': 78, 'FPCT_team': 0.963, 'HB_team': 55, 'HBP_team': 78, 'HA_team': 559, 'HAPG_team': 10.17, 'HR_team': 60, 'HRPG_team': 1.05, 'SF_team': 26, 'SH_team': 28, 'OBP_team': 0.374, 'SB_team': 29, 'SBPG_team': 0.51, 'CS_team': 15, 'R (Batting)_team': 342, 'RPGTeam': 6, 'SHO_team': 2, 'TB_team': 786, 'SLG_team': 0.423, 'SO_team': 466, 'BB (Pitching)_team': 215, 'K/BB_team': 2.17, 'K/9_team': 8.5, 'TP_team': 0, '3B_team': 8, '3BPG_team': 0.14, 'WHIP_team': 1.56, 'BBPG (Pitching)_team': 3.91,
})

# Only Pitching stats
predict_from_stats('Pitcher', 21, {
    'era_pitch': 6.50, 'fip_pitch': 1.503, 'so_pitch': 70, 'ip_pitch': 63.2,
    'bb_pitch': 50, 'h_pitch': 62, 'hr_pitch': 4, 'r_pitch': 59, 'er_pitch': 46,
    'hbp_pitch': 11, 'wp_pitch': 15, 'bk_pitch': 7, 'cg_pitch': 2, 'sho_pitch': 0,
    'g_pitch': 13, 'gs_pitch': 12, 'w_pitch': 6, 'l_pitch': 2,
    'age': 21,
}, team_stats_dict={
    'conf_rpi_team': 0.5173, 'conf_rank_team': 7, 'W_team': 28, 'L_team': 29, 'T_team': 0, 'G_team': 57, 'WPCT_team': 0.491, 'PE_team': 0.474, 'Difference_team': 0.017, 'BB (Batting)_team': 269, 'AB_team': 1857, 'H_team': 488, 'BA_team': 0.263, 'DP_team': 42, 'DPPG_team': 0.74, '2B_team': 102, '2BPG_team': 1.79, 'IP_team': 494.2, 'R (Pitching)_team': 362, 'ER_team': 320, 'ERA_team': 5.82, 'PO_team': 1484, 'A_team': 520, 'E_team': 78, 'FPCT_team': 0.963, 'HB_team': 55, 'HBP_team': 78, 'HA_team': 559, 'HAPG_team': 10.17, 'HR_team': 60, 'HRPG_team': 1.05, 'SF_team': 26, 'SH_team': 28, 'OBP_team': 0.374, 'SB_team': 29, 'SBPG_team': 0.51, 'CS_team': 15, 'R (Batting)_team': 342, 'RPGTeam': 6, 'SHO_team': 2, 'TB_team': 786, 'SLG_team': 0.423, 'SO_team': 466, 'BB (Pitching)_team': 215, 'K/BB_team': 2.17, 'K/9_team': 8.5, 'TP_team': 0, '3B_team': 8, '3BPG_team': 0.14, 'WHIP_team': 1.56, 'BBPG (Pitching)_team': 3.91,
})

# Only Batting stats
predict_from_stats('Batter', 21, {
    'age': 21, 'avg_bat': 0.310, 'ops_bat': 0.982, 'hr_bat': 2, 'slg_bat': 0.500, 'obp_bat': 0.482,
    'g_bat': 34, 'ab_bat': 42, 'h_bat': 13, '2b_bat': 2, '3b_bat': 0, 'r_bat': 12,
    'rbi_bat': 10, 'bb_bat': 14, 'so_bat': 13, 'hbp_bat': 0,
    'sf_bat': 0, 'sh_bat': 0, 'gdp_bat': 0, 'sb_bat': 0,
}, team_stats_dict={
    'conf_rpi_team': 0.5173, 'conf_rank_team': 7, 'W_team': 28, 'L_team': 29, 'T_team': 0, 'G_team': 57, 'WPCT_team': 0.491, 'PE_team': 0.474, 'Difference_team': 0.017, 'BB (Batting)_team': 269, 'AB_team': 1857, 'H_team': 488, 'BA_team': 0.263, 'DP_team': 42, 'DPPG_team': 0.74, '2B_team': 102, '2BPG_team': 1.79, 'IP_team': 494.2, 'R (Pitching)_team': 362, 'ER_team': 320, 'ERA_team': 5.82, 'PO_team': 1484, 'A_team': 520, 'E_team': 78, 'FPCT_team': 0.963, 'HB_team': 55, 'HBP_team': 78, 'HA_team': 559, 'HAPG_team': 10.17, 'HR_team': 60, 'HRPG_team': 1.05, 'SF_team': 26, 'SH_team': 28, 'OBP_team': 0.374, 'SB_team': 29, 'SBPG_team': 0.51, 'CS_team': 15, 'R (Batting)_team': 342, 'RPGTeam': 6, 'SHO_team': 2, 'TB_team': 786, 'SLG_team': 0.423, 'SO_team': 466, 'BB (Pitching)_team': 215, 'K/BB_team': 2.17, 'K/9_team': 8.5, 'TP_team': 0, '3B_team': 8, '3BPG_team': 0.14, 'WHIP_team': 1.56, 'BBPG (Pitching)_team': 3.91,
})

In [ ]:
# Test Both with team stats USF
predict_from_stats('Both', 21, {
    'era_pitch': 6.50, 'fip_pitch': 1.503, 'so_pitch': 70, 'ip_pitch': 63.2,
    'bb_pitch': 50, 'h_pitch': 62, 'hr_pitch': 4, 'r_pitch': 59, 'er_pitch': 46,
    'hbp_pitch': 11, 'bk_pitch': 7, 'cg_pitch': 2, 'sho_pitch': 0,
    'g_pitch': 13, 'gs_pitch': 12, 'w_pitch': 6, 'l_pitch': 2,
    'age': 21, 'avg_bat': 0.310, 'ops_bat': 0.982, 'hr_bat': 2, 'slg_bat': 0.500, 'obp_bat': 0.482,
    'g_bat': 34, 'ab_bat': 42, 'h_bat': 13, '2b_bat': 2, '3b_bat': 0, 'r_bat': 12,
    'rbi_bat': 10, 'bb_bat': 14, 'so_bat': 13, 'hbp_bat': 0,
    'sf_bat': 0, 'sh_bat': 0, 'gdp_bat': 0, 'sb_bat': 0,
}, name='name', team='USF', team_year=2024)

---
## Scouting Report Demos

Showcasing the new strengths / concerns / grade view across a strong projection
(Kade Anderson), a weak projection (Cooper Clark), and a custom stat line.
The existing demo cells above also now show the scouting report automatically
because `lookup_player` and `predict_from_stats` call it by default.

In [ ]:
# Strong projection -- expect A-range grades, K/9 / FIP / WHIP among strengths.
_ = lookup_player('Kade Anderson')

# Weak projection -- expect grade flip, ERA/WHIP/K% among concerns.
_ = lookup_player('cooper clark')

# Custom pitcher (elite line)
_ = predict_from_stats('Pitcher', 21, {
    'era_pitch': 2.10, 'fip_pitch': 2.40, 'so_pitch': 130, 'ip_pitch': 100.0,
    'bb_pitch': 18, 'k/9_pitch': 11.7, 'bb/9_pitch': 1.6, 'whip_pitch': 0.92,
    'k%_pitch': 0.34, 'bb%_pitch': 0.05, 'k-bb%_pitch': 0.29,
    'g_pitch': 16, 'gs_pitch': 16, 'w_pitch': 11, 'l_pitch': 1,
}, name='Hypothetical Ace')

# Custom batter (mid-tier line, expect mixed grades)
_ = predict_from_stats('Batter', 22, {
    'avg_bat': 0.290, 'ops_bat': 0.860, 'hr_bat': 10, 'wrc+_bat': 120,
    'woba_bat': 0.380, 'slg_bat': 0.480, 'obp_bat': 0.380,
    'bb%_bat': 0.10, 'k%_bat': 0.20, 'iso_bat': 0.190,
    'g_bat': 55, 'ab_bat': 205, 'pa_bat': 240, 'h_bat': 60,
    'r_bat': 40, 'rbi_bat': 40, 'sb_bat': 5,
}, name='Mid-tier Batter')

# Backwards compatibility check -- explain=False yields the pre-V7 output exactly.
print("\n\n[explain=False sanity check below: should match pre-V7 output]")
_ = predict_from_stats('Pitcher', 21, {
    'era_pitch': 2.10, 'k/9_pitch': 11.7, 'whip_pitch': 0.92,
}, explain=False)

---
## Pre-draft simulation

Simulate the moment before a completed draft and grade the whole pipeline against what actually
happened.

Unlike the sections above, which score only known draftees, this runs the entire player population
through all three stages using models that saw nothing from the simulated year, and only pre-draft
information -- the `api_*` biometrics are dropped so every player can be scored. Players are then
gated by draft probability, filtered to those eligible on pre-draft evidence alone, and ordered by
the rank model.

Set `SIM_TEST_YEAR` to change which draft is simulated.

In [ ]:
# Retrain all three stages using only what was known before the simulated draft

# SIM_TEST_YEAR, AGE_ELIGIBLE and CLASS_ELIGIBLE_SEASONS come from the config cell.
PROB_GATE = 0.90              # keep players with P(drafted) >= this on the board
BOARD_SIZE = 300              # ~ number of college players drafted per year

# Stats-only feature sets: drop post-draft api_* so the models apply to EVERY
# player (drafted or not). Stage 1 (features) is already api-free.
sim_cls_features  = [f for f in features if not str(f).startswith('api_')]
sim_rank_features = [f for f in pick_features if not str(f).startswith('api_')]
print(f"SIM_TEST_YEAR={SIM_TEST_YEAR} | classifier feats={len(sim_cls_features)} | "
      f"rank/bonus/slot feats={len(sim_rank_features)} (api_* dropped)")

# --- Stage 1: draft classifier, trained STRICTLY on years < SIM_TEST_YEAR ---
# df is already eligibility-filtered (preamble), so the sim trains on eligible players only.
_hist = df[df[year_col] < SIM_TEST_YEAR]
_pos = _hist[_hist['Drafted?'] == 1].index
_neg = _hist[_hist['Drafted?'] == 0].index
np.random.seed(40)
_neg_us = np.random.choice(_neg, size=len(_pos), replace=False)
_bal = np.concatenate([_pos.values, _neg_us]); np.random.shuffle(_bal)
draft_model_sim = xgb.XGBClassifier(**RP_CLF)
draft_model_sim.fit(df.loc[_bal, sim_cls_features], df.loc[_bal, 'Drafted?'])

# --- Stage 2: college draft order, drafted players < SIM_TEST_YEAR ---
sim_drafted_train = draftedClean_df[draftedClean_df[year_col] < SIM_TEST_YEAR]
model_rank_sim = xgb.XGBRegressor(**RP_REG)
model_rank_sim.fit(sim_drafted_train[sim_rank_features], sim_drafted_train['College_Draft_Order'])

# --- Stage 3: bonus/slot ratio + denominator, drafted players < SIM_TEST_YEAR ---
# is_post10 can't be a feature here -- before the draft nobody knows the round. The next cell
# derives it from Stage 2's predicted order, so this Stage 3 carries Stage 2's error.
_sim_ratio_features = sim_rank_features + ['is_post10']
_rtr = sim_drafted_train[sim_drafted_train['api_signingBonus'].notna()]
ratio_model_sim = xgb.XGBRegressor(**RP_REG)
ratio_model_sim.fit(_rtr[_sim_ratio_features], _rtr['log_ratio'])
slot_model_sim = xgb.XGBRegressor(**RP_REG)
slot_model_sim.fit(sim_drafted_train[sim_rank_features],
                   np.log(sim_drafted_train['slot_denom'] * 1e6))

print(f"Trained on years {sorted(_hist[year_col].unique())} "
      f"(held out: {SIM_TEST_YEAR}). "
      f"Stage1 balanced n={len(_bal)}, Stage2 n={len(sim_drafted_train)}, "
      f"ratio n={len(_rtr)}.")

In [ ]:
# ---- Score every SIM_TEST_YEAR player + build the board ----
# Scored from df_all, the unfiltered population, so the audit below can report what the
# eligibility gate removed

sim = df_all[df_all[year_col] == SIM_TEST_YEAR].copy()
sim['draft_prob'] = draft_model_sim.predict_proba(sim[sim_cls_features])[:, 1]
sim['rank_raw']   = model_rank_sim.predict(sim[sim_rank_features])

# The round is unknown pre-draft, so take the segment flag from the predicted college order
sim['pred_order_tmp'] = sim['rank_raw'].rank(method='min')
sim['is_post10'] = (sim['pred_order_tmp'] > POST10_ORDER_CUT).astype(int)
sim['pred_ratio'] = np.exp(ratio_model_sim.predict(sim[sim_rank_features + ['is_post10']]))
sim['pred_slot']  = np.exp(slot_model_sim.predict(sim[sim_rank_features]))
sim['pred_bonus'] = sim['pred_ratio'] * sim['pred_slot']

# Gate by confidence, order by the rank model, keep the pre-draft eligible, then cap.
# eligible_predraft, so the draft outcome plays no part in who makes the board.
gated = sim[sim['draft_prob'] >= PROB_GATE].sort_values('rank_raw').reset_index(drop=True)
gated['raw_board_pos'] = range(1, len(gated) + 1)
sim_board = gated[gated['eligible_predraft']].head(BOARD_SIZE).reset_index(drop=True)
sim_board['Predicted_Draft_Order'] = range(1, len(sim_board) + 1)
sim_board['Tier'] = sim_board['Predicted_Draft_Order'].map(
    lambda p: 'Early' if p <= 50 else ('Middle' if p <= 150 else 'Late'))

print(f"Scored {len(sim)} {SIM_TEST_YEAR} players (full pre-eligibility population).")
print(f"  pass gate (P >= {PROB_GATE:.0%}): {len(gated)}  |  "
      f"eligible pre-draft: {int(gated['eligible_predraft'].sum())}"
      f"  |  removed as ineligible: {int((~gated['eligible_predraft']).sum())}")
print(f"  final board: {len(sim_board)} "
      f"(pre-draft basis: {dict(sim_board['eligibility_basis_predraft'].value_counts())})")

_out = sim_board[['Predicted_Draft_Order', 'Tier', 'nameascii', 'draft_prob', 'rank_raw',
                  'pred_ratio', 'pred_bonus', 'pred_slot', 'age', 'age_filled',
                  'seasons_to_date', 'eligibility_basis_predraft']].copy()
_out['draft_prob'] = (_out['draft_prob'] * 100).round(1)
_out['pred_ratio'] = _out['pred_ratio'].round(3)
for _c in ['pred_bonus', 'pred_slot']:
    _out[_c] = _out[_c].round(0)
_out.to_csv(f'{SIM_TEST_YEAR}_simulated_board.csv', index=False)
print(f"saved -> {SIM_TEST_YEAR}_simulated_board.csv")
display(_out.head(25))

In [ ]:
# Grade the simulated board against what actually happened, and against MLB Pipeline
from sklearn.metrics import average_precision_score, roc_auc_score

_clean = lambda s: str(s).strip().lower()

# Actual SIM_TEST_YEAR outcomes (this draft already happened in the historical data).
# Rename Pick -> actual_pick: df (and thus sim_board) already carries a 'Pick' column,
# so an un-renamed merge would collide into Pick_x / Pick_y.
actual_detail = (draftedClean_df[draftedClean_df[year_col] == SIM_TEST_YEAR]
                 [['nameascii', 'Pick', 'College_Draft_Order',
                   'api_signingBonus', 'api_pickValue', 'slot_denom',
                   'bonus_slot_ratio', 'is_post10']].copy()
                 .rename(columns={'is_post10': 'actual_is_post10'})
                 .rename(columns={'Pick': 'actual_pick'}))
# is_post10 is renamed above: sim_board already carries its own (predicted) is_post10, so an
# un-renamed merge would split into is_post10_x / is_post10_y.
drafted_names = set(df[(df[year_col] == SIM_TEST_YEAR) & (df['Drafted?'] == 1)]['nameascii'])
sim['actual_drafted'] = sim['nameascii'].isin(drafted_names).astype(int)

print("=" * 72)
print(f"PRE-DRAFT SIMULATION GRADED vs REALITY — {SIM_TEST_YEAR}")
print("=" * 72)

# Stage 1 scored on the whole population, not just known draftees
y_true, y_score = sim['actual_drafted'].values, sim['draft_prob'].values
ap = average_precision_score(y_true, y_score)
auc = roc_auc_score(y_true, y_score)
pred_pos = sim['draft_prob'] >= PROB_GATE
TP = int((pred_pos & (sim['actual_drafted'] == 1)).sum())
FP = int((pred_pos & (sim['actual_drafted'] == 0)).sum())
FN = int((~pred_pos & (sim['actual_drafted'] == 1)).sum())
prec = TP / (TP + FP) if TP + FP else float('nan')
rec = TP / (TP + FN) if TP + FN else float('nan')
print(f"\nSTAGE 1 — Draftable?  (all {len(sim)} players; {int(y_true.sum())} were actually drafted)")
print(f"  PR-AUC {ap:.3f}   ROC-AUC {auc:.3f}")
print(f"  At gate P>= {PROB_GATE:.0%}: flagged {int(pred_pos.sum())}  ->  "
      f"precision {prec:.3f}  recall {rec:.3f}   (TP {TP}, FP {FP}, FN {FN})")

# Any removed player who was actually drafted is a false removal -- usually a sophomore who
# turned 21 just before the draft, which a season-snapshot age cannot see
removed = gated[~gated['eligible_predraft']].copy()
removed['actually_drafted'] = removed['nameascii'].isin(drafted_names)
n_removed, n_bad = len(removed), int(removed['actually_drafted'].sum())
print(f"\nELIGIBILITY FILTER AUDIT:")
print(f"  Removed {n_removed} gated players as ineligible.")
print(f"  Of those, {n_bad} were ACTUALLY drafted -> FALSE removals "
      f"({n_bad/n_removed*100:.0f}% of removals)." if n_removed else "  (nothing removed)")
if n_bad:
    _bad = (removed[removed['actually_drafted']]
            .merge(actual_detail, on='nameascii', how='left').sort_values('actual_pick'))
    print(f"  Actually-drafted players the filter wrongly cut (pick | name | age | seasons):")
    for _, x in _bad.iterrows():
        _pk = f"#{int(x['actual_pick'])}" if pd.notna(x['actual_pick']) else "n/a"
        print(f"    {_pk:>4}  {x['nameascii']:22} age {x['age']}  {int(x['seasons_to_date'])} seasons")

# How many of the board's top K were actually drafted, next to MLB Pipeline
prosp = prospects_by_year.get(SIM_TEST_YEAR)
prosp_names = prosp.sort_values('rank')['name_clean'].tolist() if prosp is not None else []
drafted_clean = {_clean(n) for n in drafted_names}
board_names = sim_board['nameascii'].map(_clean).tolist()
def _hits(names, k):
    top = names[:k]
    return sum(n in drafted_clean for n in top), len(top)
print(f"\nBOARD QUALITY — of the top-K names, how many were ACTUALLY drafted:")
print(f"  {'K':>4} {'your board':>14} {'MLB Pipeline':>14}")
for k in [50, 100, 200, BOARD_SIZE]:
    yb, pb = _hits(board_names, k), _hits(prosp_names, k)
    print(f"  {k:>4} {f'{yb[0]}/{yb[1]}':>14} {f'{pb[0]}/{pb[1]}':>14}")

# Stage 2 order, among board players who were actually drafted
bd = sim_board.merge(actual_detail, on='nameascii', how='left')
bd['name_clean'] = bd['nameascii'].map(_clean)
if prosp is not None:
    bd = bd.merge(prosp[['name_clean', 'rank']].rename(columns={'rank': 'pipeline_rank'}),
                  on='name_clean', how='left')
else:
    bd['pipeline_rank'] = np.nan
m2 = bd[bd['College_Draft_Order'].notna()]
print(f"\nSTAGE 2 — Draft order  (board players who were actually drafted, n={len(m2)}):")
if len(m2) >= 5:
    sp_m, _ = spearmanr(m2['College_Draft_Order'], m2['Predicted_Draft_Order'])
    mae_m = mean_absolute_error(m2['College_Draft_Order'].rank(method='min'),
                                m2['Predicted_Draft_Order'].rank(method='min'))
    print(f"  Model vs actual:         Spearman {sp_m:.3f}   MAE(rank) {mae_m:.1f}")
    mp = m2[m2['pipeline_rank'].notna()]
    if len(mp) >= 5:
        sp_p, _ = spearmanr(mp['College_Draft_Order'], mp['pipeline_rank'])
        print(f"  MLB Pipeline vs actual:  Spearman {sp_p:.3f}   (n={len(mp)})")

# Stage 3 ratio among board players who were actually drafted. is_post10 came from Stage 2's
# predicted order, so these numbers carry Stage 2's error too.
mr = bd[bd['bonus_slot_ratio'].notna()]
ms = bd[bd['slot_denom'].notna()]
print(f"\nSTAGE 3 — Bonus/slot ratio  (board players actually drafted):")
if len(mr):
    ar = mr['bonus_slot_ratio'].clip(*RATIO_CLIP)
    _dir_ok = (np.sign(np.log(ar.values)) == np.sign(np.log(mr['pred_ratio'].values)))
    print(f"  Ratio n={len(mr):>3}  MAE {mean_absolute_error(ar, mr['pred_ratio']):>8.3f}"
          f"  R2 {r2_score(ar, mr['pred_ratio']):>6.3f}"
          f"  Spearman {spearmanr(ar, mr['pred_ratio'])[0]:>6.3f}"
          f"  direction {_dir_ok.mean():>5.1%}")
    ab = mr['api_signingBonus'] * 1e6
    print(f"  Bonus n={len(mr):>3}  MAE ${mean_absolute_error(ab, mr['pred_bonus']):>11,.0f}"
          f"  R2 {r2_score(ab, mr['pred_bonus']):>6.3f}  Spearman {spearmanr(ab, mr['pred_bonus'])[0]:>6.3f}")
    _regime = (mr['actual_is_post10'].values == mr['is_post10'].values)
    print(f"  round-11+ regime guessed correctly from predicted order: {_regime.mean():.1%}")
if len(ms):
    aslot = ms['slot_denom'] * 1e6
    print(f"  Slot  n={len(ms):>3}  MAE ${mean_absolute_error(aslot, ms['pred_slot']):>11,.0f}"
          f"  R2 {r2_score(aslot, ms['pred_slot']):>6.3f}  Spearman {spearmanr(aslot, ms['pred_slot'])[0]:>6.3f}")

# Graded board for inspection (predicted next to actual).
sim_board_graded = bd[['Predicted_Draft_Order', 'Tier', 'nameascii', 'draft_prob',
                       'pred_ratio', 'pred_bonus', 'eligibility_basis_predraft', 'actual_pick',
                       'College_Draft_Order', 'pipeline_rank', 'api_signingBonus',
                       'bonus_slot_ratio']].copy()
sim_board_graded['actually_drafted'] = sim_board_graded['nameascii'].isin(drafted_names)
sim_board_graded['draft_prob'] = (sim_board_graded['draft_prob'] * 100).round(1)
print(f"\nGraded board (predicted vs actual); actually_drafted=False rows are pre-draft 'misses':")
pd.set_option('display.max_rows', 60)
display(sim_board_graded.head(50))

In [ ]:
# Walk a single player through all three pre-draft models
# Walk ANY SIM_TEST_YEAR player (drafted or not) through all 3 sim stages and line
# it up against the real outcome and the MLB Pipeline rank.
def _money2(x):
    return f"${x:,.0f}" if x is not None and pd.notna(x) else "n/a"

def sim_trace(name_query, verbose=True):
    names = sim['nameascii'].dropna().unique().tolist()
    m = get_close_matches(name_query, names, n=1, cutoff=0.4)
    if not m:
        if verbose: print(f"No {SIM_TEST_YEAR} player match for '{name_query}'")
        return None
    r = sim[sim['nameascii'] == m[0]].iloc[0]
    nm = r['nameascii']
    role_name = {0: 'Pitcher', 1: 'Batter', 2: 'Two-Way'}.get(r['role'], '?')
    act = actual_detail[actual_detail['nameascii'] == nm]
    was_drafted = nm in drafted_names
    prosp = prospects_by_year.get(SIM_TEST_YEAR)
    pr = prosp[prosp['name_clean'] == str(nm).strip().lower()] if prosp is not None else None
    pipe_rank = int(pr.iloc[0]['rank']) if pr is not None and len(pr) else None
    on_board = sim_board[sim_board['nameascii'] == nm]
    board_pos = int(on_board.iloc[0]['Predicted_Draft_Order']) if len(on_board) else None

    if verbose:
        L = "=" * 68
        print(L)
        print(f"PRE-DRAFT TRACE — {nm}  ({SIM_TEST_YEAR})  |  {role_name}  "
              f"|  eligibility: {r['eligibility_basis']}")
        print(L)
        print("STAGE 1 — Draftable?")
        print(f"  Model P(drafted):   {r['draft_prob']*100:5.1f}%   "
              f"(gate {PROB_GATE:.0%}: {'PASS' if r['draft_prob']>=PROB_GATE else 'below'})")
        print(f"  Real:               {'DRAFTED' if was_drafted else 'NOT drafted'}"
              + (f" — overall pick #{int(act.iloc[0]['actual_pick'])}" if was_drafted and len(act) else ""))
        print(f"  MLB Pipeline:       " + (f"#{pipe_rank} of top-250" if pipe_rank else "not in top-250"))
        if not r['eligible_predraft']:
            print(f"  -> removed from board (draft-ineligible: {r['eligibility_basis_predraft']}, "
                  f"age {r['age']}, {int(r['total_college_seasons'])} college seasons)")
        else:
            print("STAGE 2 — Draft order")
            print(f"  Board position:     " + (f"#{board_pos}" if board_pos else "not on board (missed gate)"))
            ao = int(act.iloc[0]['College_Draft_Order']) if was_drafted and len(act) else None
            print(f"  Real college order: " + (f"{ao}" if ao else "n/a (not drafted)"))
            print(f"  MLB Pipeline order: " + (f"{pipe_rank}" if pipe_rank else "n/a"))
            print("STAGE 3 — Signing bonus & slot value")
            ab = act.iloc[0]['api_signingBonus'] * 1e6 if was_drafted and len(act) and pd.notna(act.iloc[0]['api_signingBonus']) else None
            aslot = act.iloc[0]['slot_denom'] * 1e6 if was_drafted and len(act) and pd.notna(act.iloc[0]['slot_denom']) else None
            print(f"  Model bonus:        {_money2(r['pred_bonus']):>13}   Real: {_money2(ab):>13}")
            print(f"  Model slot:         {_money2(r['pred_slot']):>13}   Real: {_money2(aslot):>13}")
            _ar = act.iloc[0]['bonus_slot_ratio'] if was_drafted and len(act) else None
            print(f"  Model bonus/slot:   {r['pred_ratio']:>13.3f}   Real: "
                  f"{('%.3f' % _ar) if pd.notna(_ar) else 'n/a':>13}")
        print(L)
    return r

print("sim_trace() defined. Usage: sim_trace('Kade Anderson')\n")

# Demo: top-3 board players, plus the top few high-confidence players REMOVED for
# ineligibility (the filter in action), plus a board 'miss' if one exists.
for _nm in sim_board['nameascii'].head(3).tolist():
    sim_trace(_nm)

_removed = gated[~gated['eligible_predraft']].head(5)
if len(_removed):
    print("\nTop high-confidence players REMOVED by the eligibility filter "
          "(would-be board spot -> reason):")
    for _, _r in _removed.iterrows():
        print(f"  raw #{int(_r['raw_board_pos']):>3}  {_r['nameascii']:22} "
              f"P={_r['draft_prob']*100:4.1f}%  age {_r['age']}  "
              f"{int(_r['total_college_seasons'])} seasons  ({_r['eligibility_basis']})")
    _ex = _removed.iloc[0]['nameascii']
    print()
    sim_trace(_ex)

## Carbon Tracking

In [ ]:
emissions = tracker.stop()

# Constants for equivalencies
CAR_EMISSION_FACTOR = 0.12        # kgCO2 per km
TV_DAILY_WH = 138                 # Wh per day
TV_EMISSION_FACTOR = 0.084        # kgCO2 per kWh (implied by source)
US_WEEKLY_KG = (13.3 * 1000) / 52  # kg CO2 per week for avg US citizen

energy_kwh = tracker.final_emissions_data.energy_consumed

print("\nCarbon Emissions Summary:")
print(f"  Total CO₂ emitted:     {emissions:.7f} kg")
print(f"  Total energy consumed: {energy_kwh:.7f} kWh")
print(f"  Total cost of energy:  ${tracker.final_emissions_data.energy_consumed*0.269141194:.7f} USD")
print(f"    CPU energy:          {tracker.final_emissions_data.cpu_energy:.7f} kWh")
print(f"    RAM energy:          {tracker.final_emissions_data.ram_energy:.7f} kWh")
_gpu_kwh = tracker.final_emissions_data.gpu_energy
def _nvidia_gpu_present():
    try:
        pynvml.nvmlInit()
        n = pynvml.nvmlDeviceGetCount()
        pynvml.nvmlShutdown()
        return n > 0
    except Exception:
        return False
if _gpu_kwh == 0 and not _nvidia_gpu_present():
    print(f"    GPU energy:          N/A (no NVIDIA GPU detected by codecarbon)")
else:
    print(f"    GPU energy:          {_gpu_kwh:.7f} kWh")
print(f"  Duration:              {tracker.final_emissions_data.duration:.2f} s")

print("\nEquivalency Estimates:")
print(f"  🚗 Car distance:       {emissions / CAR_EMISSION_FACTOR:.4f} km driven")
print(f"  📺 TV usage:           {(energy_kwh / (TV_DAILY_WH / 1000)):.4f} days of TV")
print(f"  🧍 US citizen:         {(emissions / US_WEEKLY_KG) * 100:.6f}% of weekly per-capita emissions")

# Cumulative stats from emissions.csv
EMISSIONS_CSV = os.path.join(os.getcwd(), "emissions.csv")

if os.path.exists(EMISSIONS_CSV):
    emissions_df = pd.read_csv(EMISSIONS_CSV)
    total_emissions = emissions_df["emissions"].sum()
    total_energy    = emissions_df["energy_consumed"].sum()
    total_duration  = emissions_df["duration"].sum()
    n_runs          = len(emissions_df)

    print(f"\nCumulative Summary ({n_runs} runs):")
    print(f"  Total CO₂ emitted:     {total_emissions:.7f} kg")
    print(f"  Total energy consumed: {total_energy:.7f} kWh")
    print(f"  Total cost of energy:  ${total_energy * 0.269141194:.7f} USD")
    print(f"  Total duration:        {total_duration:.2f} s ({total_duration/3600:.2f} hrs)")

    print("\nEquivalency Estimates (Cumulative):")
    print(f"  🚗 Car distance:       {total_emissions / CAR_EMISSION_FACTOR:.4f} km driven")
    print(f"  📺 TV usage:           {(total_energy / (TV_DAILY_WH / 1000)):.4f} days of TV")
    print(f"  🧍 US citizen:         {(total_emissions / US_WEEKLY_KG) * 100:.6f}% of weekly per-capita emissions")